# Reproducing Petersen & Buzsaki (2020) from DANDI:000059

Paper: Petersen PC, Buzsaki G (2020). Cooling of medial septum reveals theta phase lag coordination of hippocampal cell assemblies. Neuron 107, 731-744. doi:10.1016/j.neuron.2020.05.023.

Dataset: DANDI:000059, version 0.261115.0311 (doi:10.48324/dandi.000059/0.261115.0311). The paper cites no version of the dandiset. This is the most recent published version; it holds the 51 sessions of 5 rats as one file per session, with the lab's trial cooling states, the measured MS temperature where it exists, the linearized position and the cell types, which the analyses use. The earlier versions (0.210812.1514, 0.230907.2101, 0.250624.0444) split each session into raw and processed files and, by the new version's release notes, had wrong electrode groups and trial times one sample late; none matches the claims file's description better.

Claims: `claims/petersen-2020.yaml`, written from the accepted author manuscript in PubMed Central (PMC7442698) with the STAR Methods and the Supplemental Information (Figures S1 to S6). Protocol pilot-15.

Data are streamed from the archive; nothing is written to disk. Loading reduces each session to its trials, head tracking, MS temperature, the theta of one CA1 LFP channel during the trials, and the spikes of the curated units during the trials.

The table of claims and outcomes is in the Summary at the end; which claims are tested and which blocked is listed here.

| claim | tested here |
|---|---|
| session-structure, cooling-time-course | tested |
| fig2g, fig2h, fig2j, fig2i, disc-theta-frequency-12-percent | tested |
| fig6d, fig6e, fig6c, fig6f, fig6g, fig6h, fig6i, suppl5j, fig4e, fig5-speed-control-cooling | tested |
| suppl2-place-cell-rates-decrease, suppl2ab, disc-interneuron-rates-decrease, suppl5gh | tested |
| fig4b, fig4c, fig4ac, fig4a, fig3c, fig3d, fig3e, fig3f, fig3hi, fig4d, suppl3e, fig4f, fig4gh, suppl2e, suppl3 | tested |
| fig5d, fig5e, fig5h, fig5i, fig5j, fig5k, suppl2d | tested |
| fig5cg-compression | blocked, methods_underspecified: the surface fit does not describe the density (see its section) |
| suppl1g-waveforms-unchanged | see its section |
| suppl1-hippocampal-temperature | blocked, data_missing: no hippocampal temperature in the dandiset |

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
import time, resource, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats
import ms_cooling as mc
warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda v: f"{v:.4g}")
T_START = time.time()

In [ ]:
sessions = mc.list_sessions()
print(f"dandiset {mc.DANDISET} version {mc.VERSION}: {len(sessions)} NWB files")
print(sessions.groupby("rat").size().to_string())
SESS = mc.load_all(sessions, workers=4)
print(f"loaded in {time.time() - T_START:.0f} s; peak memory of this process "
      f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

## Data check

In [ ]:
rows = []
for S in SESS:
    tr = S["trials"]
    tt, T = S["temp"]
    U = S.get("units", [])
    ct = pd.Series([u["cell_type"] for u in U]).value_counts().to_dict() if U else {}
    rows.append(dict(rat=S["rat"], session=S["session"], trials=len(tr),
                     pre=(tr.state == "Pre").sum(), cooling=(tr.state == "Cooling").sum(),
                     post=(tr.state == "Post").sum(), transition=(tr.state == "transition (excluded)").sum(),
                     other=(~tr.state.isin(["Pre", "Cooling", "Post", "transition (excluded)"])).sum(),
                     errors=int(tr.error.sum()), cooling_epochs=len(S["onsets"]),
                     temp=S["temp_kind"], temp_min=np.nanmin(T), temp_max=np.nanmax(T),
                     pos_hz=round(1 / S["pos_dt"], 1), lfp=S["has_lfp"],
                     theta_ch=S.get("theta_info", {}).get("channel"),
                     theta_ch_bad=S.get("theta_info", {}).get("theta_tag_bad"),
                     good_units=len(U), pyr=ct.get("Pyramidal Cell", 0),
                     nint=ct.get("Narrow Interneuron", 0), wint=ct.get("Wide Interneuron", 0),
                     untyped=ct.get("", 0) + ct.get("Unknown", 0)))
SAN = pd.DataFrame(rows)
print(SAN.to_string())
print()
print("sessions per rat:", SAN.groupby("rat").size().to_dict())
print("sessions with measured MS temperature:", (SAN.temp == "measured").sum(),
      "; modelled (not a measurement):", (SAN.temp == "modelled").sum())
print("sessions with LFP:", SAN.lfp.sum(), "; with curated units:", (SAN.good_units > 0).sum(),
      "; with typed pyramidal cells:", (SAN.pyr > 0).sum())
print("sessions with two cooling epochs:", (SAN.cooling_epochs > 1).sum())
print("tracking rate (Hz) by session:", SAN.pos_hz.value_counts().to_dict(),
      "; the descriptions say 40 Hz:", sorted(set(S["beh_desc"].split("OptiTrack head tracking at ")[1].split(" Hz")[0] for S in SESS)))
allspd = np.concatenate([S["beh"]["speed"] for S in SESS])
alllin = np.concatenate([S["beh"]["lin"] for S in SESS])
print(f"speed (cm/s): 1st, 50th, 99th percentiles {np.nanpercentile(allspd, [1, 50, 99]).round(1)}; "
      f"linearized position (cm): {np.nanmin(alllin):.1f} to {np.nanmax(alllin):.1f}")
alltr = pd.concat([S["trials"].assign(session=S["session"]) for S in SESS])
print(f"trial duration (s): median {np.median(alltr.stop - alltr.start):.2f}, range "
      f"{(alltr.stop - alltr.start).min():.2f} to {(alltr.stop - alltr.start).max():.2f}")
print("arm run (from the linearized position) against the trial label 'condition', by error:")
print(pd.crosstab([alltr.error, alltr.condition], alltr.arm))
th = alltr.dropna(subset=["f_hil"])
print(f"trial theta frequency (Hilbert, Hz): {th.f_hil.min():.2f} to {th.f_hil.max():.2f}, median {th.f_hil.median():.2f}")
print(f"trial MS temperature (C), measured sessions: "
      f"{alltr[alltr.session.isin(SAN.session[SAN.temp == 'measured'])].temp.describe()[['min', '50%', 'max']].round(2).to_dict()}")
MEAS_ALL = [S for S in SESS if S["temp_kind"] == "measured"]
for S in MEAS_ALL:
    if True:
        tt, T = S["temp"]
        raw_bad = np.isnan(T).sum()
        if raw_bad:
            print(f"  {S['session']}: {raw_bad} temperature samples outside 5 to 45 C set to NaN")
# rails near 5 and 10 C fall inside 5 to 45 C; check that no sample within the trials falls below 15 C
low = {}
for S in MEAS_ALL:
    tt, T = S["temp"]
    tr = S["trials"]
    w = (tt >= tr.start.min()) & (tt <= tr.stop.max())
    low[S["session"]] = (int(np.sum(T[w] < 15)), int(np.sum(T < 15)))
print("measured temperature samples below 15 C (within the trials, in the whole trace):",
      {k: v for k, v in low.items() if v[1] > 0}, "; sessions with any within the trials:", sum(v[0] > 0 for v in low.values()))
print("lowest trial temperature per measured session (C):",
      pd.Series({S["session"]: S["trials"].temp.min() for S in MEAS_ALL}).round(1).sort_values().head(5).to_dict())
ep = [S for S in SESS if "epochs" in S]
envs = pd.concat([S["epochs"].assign(session=S["session"]) for S in ep])
print("epoch environments and paradigms:")
print(envs.groupby(["behavioral_paradigm", "environment"]).size())

Checks of the description against the data:

- Tracking rate. Every behaviour description says "OptiTrack head tracking at 40 Hz", and the paper says 120 Hz; the timestamps give 40 Hz only in some MS10 sessions and about 120 Hz elsewhere (table above). The data are trusted.
- Temperature. Sessions without a measured septal trace hold the lab's modelled curve, labelled "NOT A MEASUREMENT"; every analysis of temperature uses only the measured traces. Some measured traces hold artefacts (values printed above).
- Arms. In correct trials the arm run (from the linearized position) is the trial's `condition`, and in error trials the other arm, as the trial table's description says (table above).
- Clock. The streams share the recording clock by their descriptions. The contrast below checks it: theta frequency of the trials is lower in cooling than in pre in nearly every session, and the decrease is aligned on the dataset's cooling onset (section fig6c); errors rise after the onset. A shift of the labels by the length of the cooling block removes the contrast (printed next).

In [ ]:
# label-shift control of the trial clock: theta frequency contrast cooling - pre with true labels and
# with the cooling block moved 30 trials later
d_true, d_shift = [], []
for S in SESS:
    tr = S["trials"]
    if "f_hil" not in tr:
        continue
    lab = tr.state.values
    d_true.append(tr.f_hil[lab == "Cooling"].mean() - tr.f_hil[lab == "Pre"].mean())
    sh = np.roll(lab, 30)
    d_shift.append(tr.f_hil[sh == "Cooling"].mean() - tr.f_hil[sh == "Pre"].mean())
print(f"cooling - pre theta frequency: true labels median {np.nanmedian(d_true):.3f} Hz "
      f"({np.sum(np.array(d_true) < 0)} of {len(d_true)} sessions negative); labels shifted 30 trials median "
      f"{np.nanmedian(d_shift):.3f} Hz ({np.sum(np.array(d_shift) < 0)} of {len(d_shift)} negative)")

## Shared definitions

- Sets of sessions. `MEAS`: sessions with a measured MS temperature trace (31; the 20 others hold the lab's modelled curve, which is not used). `LFPS`: sessions with LFP (50). The two sessions of 2018-07-19 (MS21 and MS22) are labelled as cooled, but their measured temperature does not fall; they are kept, as the lab's labels are, and results without them are shown where they could matter.
- States, the readings of the claims file's opening comment: `lab` the dataset's trial labels (Pre, Cooling, Post; transition trials left out); `m12` and `m10` cooling is the trials that start within 12 or 10 min after cooling onset. Cooling onset is the time stamp in the dataset's cooling table (the first epoch where a session has two).
- Trial theta: `f_hil`/`p_hil` the mean Hilbert instantaneous frequency and mean squared amplitude over the trial (reading a), `f_spec`/`p_spec` the peak of the trial's 4 to 10 Hz periodogram (reading b); `_run` versions use only samples above 10 cm/s.
- Aligned trials: first trial after onset is 1, the last before is -1; trials after a second cooling onset are left out of aligned series. The aligned series spans trials -40 to 80 (standard choice; the paper's Figure 6C shows about this range). Fewer sessions contribute at its ends than near the onset (for the measured sessions with LFP, 11 at the fewest, against 31; printed in fig2g).

In [ ]:
MEAS = [S for S in SESS if S["temp_kind"] == "measured"]
MEAS_NAMES = {S["session"] for S in MEAS}
LFPS = [S for S in SESS if S["has_lfp"]]
# the lab's theta channel of MS13-171130 is marked bad in the electrodes table; results without it are shown
BAD_THETA = {S["session"] for S in LFPS if any(S["theta_info"].get("theta_tag_bad", []))}
LFPS_GOOD = [S for S in LFPS if S["session"] not in BAD_THETA]
FLAT = {"MS21-180719-155941", "MS22-180719-122813"}
STATE_READINGS = ["lab", "m12", "m10"]
THETA_F = ["f_hil", "f_spec", "f_hil_run"]
THETA_P = ["p_hil", "p_spec", "p_hil_run"]
for S in SESS:
    tr = S["trials"]
    for r in STATE_READINGS:
        tr["st_" + r] = mc.state_labels(tr, S["onset"], r).values
    if S["has_lfp"]:
        pre = tr.st_lab == "Pre"
        for c in THETA_P:
            tr[c + "_norm"] = tr[c] / tr.loc[pre, c].mean()
    tr["err"] = tr.error.astype(float)
    tr["trial_no"] = np.arange(len(tr))
print("sessions whose tagged theta channel is marked bad:", sorted(BAD_THETA))
print(f"MEAS {len(MEAS)} sessions ({pd.Series([S['rat'] for S in MEAS]).value_counts().to_dict()}), "
      f"LFPS {len(LFPS)} sessions")


def fmt_p(p):
    return f"{p:.2g}"


def aligned(SS, col, k=(-40, 80)):
    a, d = mc.aligned_series(SS, col, k_range=k)
    return a

## session-structure

> A session holds about 40 control trials, then about 20 to 60 cooling trials and 20 to 80 recovery trials, 80 to 200 trials in all with a mean of 150.

In [ ]:
ntr = SAN.trials.values
print(f"sessions {len(ntr)}: trials per session mean {ntr.mean():.1f}, median {np.median(ntr):.0f}, "
      f"range {ntr.min()} to {ntr.max()}; sessions outside 72 to 220: {np.sum((ntr < 72) | (ntr > 220))} "
      f"({', '.join(SAN.session[(ntr < 72) | (ntr > 220)])})")
res = []
for r in STATE_READINGS:
    c = np.array([(S["trials"]["st_" + r] == "Cooling").sum() for S in SESS])
    p = np.array([(S["trials"]["st_" + r] == "Pre").sum() for S in SESS])
    q = np.array([(S["trials"]["st_" + r] == "Post").sum() for S in SESS])
    res.append(dict(reading=r, median_pre=np.median(p), median_cooling=np.median(c), median_post=np.median(q),
                    cooling_in_18_66=18 <= np.median(c) <= 66, post_in_18_88=18 <= np.median(q) <= 88))
print(pd.DataFrame(res).to_string(index=False))
print("by rat, mean trials:", SAN.groupby("rat").trials.mean().round(1).to_dict())
print(f"without MS10-170307 (37 trials): mean {ntr[ntr >= 72].mean():.1f}")

**Result: partially reproduced.** The 51 sessions hold 142.4 trials on average (135 to 165 allowed), and the median numbers of pre (39), cooling (45, 49 or 41 by state reading) and post trials (46, 50 or 58) fall in the allowed ranges, but one session, MS10-170307, has 37 trials, outside 72 to 220.

## cooling-time-course

> MS cooling typically peaked about 60 s after its onset and lasted about 10 to 12 minutes, about 50 trials.

In [ ]:
rows = []
for S in MEAS:
    tt, T = S["temp"]
    tr = S["trials"]
    on = S["onset"]
    pre_med = np.nanmedian(T[(tt >= tr.start.min()) & (tt < on)])
    w = (tt >= on) & (tt <= on + 300) & np.isfinite(T)
    tmin_t = tt[w][np.nanargmin(T[w])] - on
    tmin = np.nanmin(T[w])
    # (a) time until the temperature is again within 1 C of the pre median, after the minimum
    after = (tt > on + tmin_t) & np.isfinite(T)
    back = np.where(after & (T >= pre_med - 1))[0]
    dur_a = (tt[back[0]] - on) / 60 if len(back) else np.nan
    # (b) the lab's cooling interval: onset to the end of the first 'cooling' interval
    ct = S["cooling_table"]
    c1 = ct[(ct.label == "cooling") & (ct.start >= on - 1)].iloc[0]
    dur_b = (c1.stop - on) / 60
    ncool = int(((tr.start >= on) & (tr.start <= c1.stop) & (tr.state == "Cooling")).sum())
    ncool_a = int(((tr.start >= on) & (tr.start <= on + dur_a * 60)).sum()) if np.isfinite(dur_a) else np.nan
    rows.append(dict(rat=S["rat"], session=S["session"], pre_median=pre_med, minimum=tmin,
                     t_min_s=tmin_t, dur_a_min=dur_a, dur_b_min=dur_b, cooling_trials_lab=ncool,
                     trials_in_dur_a=ncool_a))
CT = pd.DataFrame(rows)
print(CT.round(2).to_string(index=False))
med = CT.drop(columns=["rat", "session"]).median()
print("medians:", med.round(2).to_dict())
CT_nf = CT[~CT.session.isin(FLAT)]
print("medians without the two flat 2018-07-19 sessions:", CT_nf.drop(columns=["rat", "session"]).median().round(2).to_dict())
print(f"time to minimum within 40.5 to 66 s: {40.5 <= med.t_min_s <= 66}; "
      f"duration (a) {med.dur_a_min:.1f} min, (b) {med.dur_b_min:.1f} min, within 9 to 13.2 or 13.5 to 16.5: "
      f"(a) {(9 <= med.dur_a_min <= 13.2) or (13.5 <= med.dur_a_min <= 16.5)}, (b) {(9 <= med.dur_b_min <= 13.2) or (13.5 <= med.dur_b_min <= 16.5)}; "
      f"cooling trials (lab) {med.cooling_trials_lab:.0f}, within 45 to 55 or 18 to 66: {(45 <= med.cooling_trials_lab <= 55) or (18 <= med.cooling_trials_lab <= 66)}")
print("by rat, median time to minimum (s):", CT.groupby("rat").t_min_s.median().round(1).to_dict())

**Result: partially reproduced.** Over the 31 sessions with a measured MS temperature, the median time from cooling onset to the minimum is 109.7 s, outside 40.5 to 66 s. The median time until the temperature is again within 1 C of the pre median is 11.7 min (within 9 to 13.2 min), and the median number of trials in the lab's first cooling interval is 32 (within the Results' 18 to 66); the lab's cooling interval itself lasts 8.1 min.

## fig2g-theta-freq-temperature, fig2h-theta-power-temperature

> Theta frequency decreased linearly with MS temperature, down to 20 C (R = 0.81, P = 5.4e-48).

> Theta power also decreased with MS temperature, less tightly than frequency (R = 0.54, P = 1.2e-16).

Readings (claims file): (a) aligned trials averaged over sessions; (b) 1 C temperature bins, R over bin means; (c) all trials pooled. Measured-temperature sessions with LFP.

In [ ]:
MEAS_L = [S for S in MEAS if S["has_lfp"]]


def temp_corr(SS, col):
    out = {}
    a = mc.aligned_series(SS, col)[0]
    t = mc.aligned_series(SS, "temp")[0]
    j = a.index.intersection(t.index)
    out["a_aligned"] = mc.corr(t.loc[j, "mean"], a.loc[j, "mean"])
    d = pd.concat([S["trials"][["temp", col]] for S in SS]).dropna()
    d["bin"] = np.floor(d.temp)
    g = d.groupby("bin")[col].agg(["mean", "count"])
    g = g[g["count"] >= 5]
    out["b_bins"] = mc.corr(g.index.values, g["mean"].values)
    out["c_pooled"] = mc.corr(d.temp, d[col])
    return out


def corr_table(SS, cols):
    rows = []
    for c in cols:
        for k, (r, p, n) in temp_corr(SS, c).items():
            rows.append(dict(measure=c, reading=k, R=r, P=p, n=n))
    return pd.DataFrame(rows)

In [ ]:
G2 = corr_table(MEAS_L, THETA_F)
G2["R_in_tol"] = (G2.R - 0.81).abs() <= 0.081
G2["P<1e-10"] = G2.P < 1e-10
print(G2.to_string(index=False))
print("without the two flat sessions:")
print(corr_table([S for S in MEAS_L if S["session"] not in FLAT], ["f_hil", "f_spec"]).to_string(index=False))
a = mc.aligned_series(MEAS_L, "f_hil")[0]
t = mc.aligned_series(MEAS_L, "temp")[0]
print(f"aligned series: trials {a.index.min()} to {a.index.max()}, sessions per trial {a.n.min()} to {a.n.max()}; "
      f"mean temperature {t['mean'].min():.1f} to {t['mean'].max():.1f} C; theta {a['mean'].min():.2f} to {a['mean'].max():.2f} Hz")
d = pd.concat([S["trials"][["temp", "f_hil"]] for S in MEAS_L]).dropna()
fit = stats.linregress(d.temp, d.f_hil)
print(f"pooled trials: slope {fit.slope:.3f} Hz/C, lowest trial temperature {d.temp.min():.1f} C")
fig, ax = plt.subplots(1, 2, figsize=(9, 3.5))
ax[0].scatter(d.temp, d.f_hil, s=2, alpha=0.3)
ax[0].set_xlabel("MS temperature (C)"); ax[0].set_ylabel("theta frequency (Hz)")
ax[1].errorbar(t.loc[a.index, "mean"], a["mean"], yerr=a.se, fmt="o", ms=3)
ax[1].set_xlabel("mean MS temperature (C)"); ax[1].set_title("aligned trials")
plt.tight_layout(); plt.show()
print(f"plotted: all {len(d)} trials and {len(a)} aligned trials; no point outside the axes")

**Result: partially reproduced.** Theta frequency rises with MS temperature under every reading with P below 1e-10, but no R is within 0.729 to 0.891: aligned trials give R = 0.978 (Hilbert, 120 points), temperature bins 0.959, pooled trials 0.554 (4313 trials); the spectral reading gives 0.935, 0.929 and 0.481. The pooled slope is 0.063 Hz per C and trial temperatures reach 17.1 C.

In [ ]:
H2 = corr_table(MEAS_L, THETA_P + [c + "_norm" for c in THETA_P])
H2["R_in_tol"] = (H2.R - 0.54).abs() <= 0.054
H2["P<1e-5"] = H2.P < 1e-5
fr = G2.set_index(["measure", "reading"]).R
H2["freq_R_same_reading"] = [fr.get((m.replace("p_", "f_").replace("_norm", ""), k), np.nan) for m, k in zip(H2.measure, H2.reading)]
H2["below_freq_R"] = H2.R < H2.freq_R_same_reading
print(H2.to_string(index=False))

**Result: reproduced.** Under reading (a), aligned trials with the mean squared Hilbert amplitude, R = 0.495 (P = 8.9e-9, 120 points), within 0.486 to 0.594 and below the frequency R of 0.978 under the same reading. The normalised power gives R = 0.489. The spectral peak power gives larger R (0.761) and pooled trials much smaller ones (0.055).

## fig2j-frequency-consistent-across-animals

> The relationship of theta frequency with MS temperature was more consistent across animals than that of theta power.

In [ ]:
rows = []
for rat in sorted(set(S["rat"] for S in MEAS_L)):
    SS = [S for S in MEAS_L if S["rat"] == rat]
    for c in ["f_hil", "p_hil", "p_hil_norm", "f_spec", "p_spec_norm"]:
        r, p, n = temp_corr(SS, c)["a_aligned"]
        rows.append(dict(rat=rat, sessions=len(SS), measure=c, R=r, P=p, n=n))
J2 = pd.DataFrame(rows)
print(J2.pivot(index="rat", columns="measure", values="R").round(3))
print(J2.pivot(index="rat", columns="measure", values="P").map(fmt_p))
for c in ["f_hil", "p_hil", "p_hil_norm", "f_spec", "p_spec_norm"]:
    v = J2[J2.measure == c]
    print(f"{c}: range of R {v.R.max() - v.R.min():.3f}, SD {v.R.std():.3f}, rats with P < 0.05 {int((v.P < 0.05).sum())} of {len(v)}, all positive {bool((v.R > 0).all())}")
print("MS10 has no measured temperature, so 4 of the 5 rats can be tested.")

**Result: reproduced.** In each of the 4 rats with a measured temperature, theta frequency correlates positively with MS temperature (R 0.559 to 0.958, every P below 4e-11), and the range of R over rats is 0.399 for frequency against 0.881 for raw power and 0.890 for normalised power (SD 0.186 against 0.396 and 0.367). MS10 has only the modelled temperature curve, so the fifth rat cannot be tested.

## fig2i-theta-pre-cooling-post

> MS cooling induced a fast and reversible decrease of hippocampal theta frequency and power, lower during cooling than before and after it.

In [ ]:
rows = []
for r, fc, pc, sset in [(r, fc, pc, "all") for r in STATE_READINGS for fc, pc in [("f_hil", "p_hil"), ("f_spec", "p_spec"), ("f_hil_run", "p_hil_run")]] + \
        [("lab", fc, pc, "without MS13-171130") for fc, pc in [("f_hil", "p_hil"), ("f_spec", "p_spec")]]:
    if True:
        M = []
        for S in (LFPS if sset == "all" else LFPS_GOOD):
            tr = S["trials"]
            g = tr.groupby("st_" + r)[[fc, pc]].mean()
            if not {"Pre", "Cooling", "Post"} <= set(g.index):
                continue
            M.append(dict(fpre=g.loc["Pre", fc], fcool=g.loc["Cooling", fc], fpost=g.loc["Post", fc],
                          ppre=g.loc["Pre", pc], pcool=g.loc["Cooling", pc], ppost=g.loc["Post", pc]))
        M = pd.DataFrame(M)
        rows.append(dict(states=r, theta=fc, sessions=len(M), set=sset,
                         f_pre=M.fpre.mean(), f_cool=M.fcool.mean(), f_post=M.fpost.mean(),
                         P_f_cool_pre=mc.signrank(M.fcool, M.fpre)[0], P_f_cool_post=mc.signrank(M.fcool, M.fpost)[0],
                         P_p_cool_pre=mc.signrank(M.pcool, M.ppre)[0], P_p_cool_post=mc.signrank(M.pcool, M.ppost)[0],
                         f_lower_both=bool((M.fcool < M.fpre).mean() > 0.5 and (M.fcool < M.fpost).mean() > 0.5),
                         p_cool_over_pre=np.median(M.pcool / M.ppre), p_cool_over_post=np.median(M.pcool / M.ppost)))
I2 = pd.DataFrame(rows)
I2["all_four_P<0.05_lower"] = (I2[["P_f_cool_pre", "P_f_cool_post", "P_p_cool_pre", "P_p_cool_post"]] < 0.05).all(1) & (I2.p_cool_over_pre < 1) & (I2.p_cool_over_post < 1) & I2.f_lower_both
print(I2.to_string(index=False))

**Result: reproduced.** Under the spectral theta reading, cooling lowers theta frequency against pre and post and theta power against pre and post (all four signed-rank P below 1e-4 over 50 sessions, power ratios 0.82 and 0.87) under each of the three state readings. Under the Hilbert reading frequency falls (8.29 to 7.56 Hz, back to 8.36) but power during cooling does not differ from post (P = 0.42). Without MS13-171130, whose tagged theta channel is marked bad, the spectral reading gives the same result (49 sessions, all four P below 1e-5).

## disc-theta-frequency-12-percent

> During MS cooling LFP theta frequency decreased by about 12%.

In [ ]:
rows = []
for r in STATE_READINGS:
    for fc in THETA_F:
        A, B = [], []
        for S in LFPS:
            g = S["trials"].groupby("st_" + r)[fc].mean()
            if not {"Pre", "Cooling", "Post"} <= set(g.index):
                continue
            A.append(100 * (1 - g["Cooling"] / g["Pre"]))
            B.append(100 * (1 - g["Cooling"] / ((g["Pre"] + g["Post"]) / 2)))
        rows.append(dict(states=r, theta=fc, a_vs_pre=np.mean(A), b_vs_pre_post=np.mean(B)))
for fc in THETA_F:
    al = aligned(LFPS, fc)
    pre = al.loc[al.index < 0, "mean"].mean()
    rows.append(dict(states="aligned", theta=fc, c_largest_aligned=100 * (1 - al.loc[al.index > 0, "mean"].min() / pre)))
D12 = pd.DataFrame(rows)
print(D12.to_string(index=False))
vals = D12[["a_vs_pre", "b_vs_pre_post", "c_largest_aligned"]].values.ravel()
vals = vals[np.isfinite(vals)]
print(f"readings within 10.8 to 13.2%: {int(np.sum((vals >= 10.8) & (vals <= 13.2)))} of {len(vals)}; all show a decrease: {bool((vals > 0).all())}")

**Result: reproduced.** Reading (c), the largest decrease of the aligned mean series relative to its pre mean, gives 11.1% (Hilbert, within 10.8 to 13.2%) and 12.9% (spectral). Readings (a) and (b), the mean per-session decrease against pre or against pre and post, give 6.6 to 9.9%.

## fig6d-errors-increase

> MS cooling increased choice errors about 3-fold, within each rat (P < 0.05 for all animals) and across all animals (P = 1e-6).

In [ ]:
def error_rates(SS, r):
    rows = []
    for S in SS:
        tr = S["trials"]
        s = tr["st_" + r]
        rows.append(dict(rat=S["rat"], session=S["session"],
                         pre=100 * tr.err[s == "Pre"].mean(), cool=100 * tr.err[s == "Cooling"].mean(),
                         post=100 * tr.err[s == "Post"].mean(),
                         prepost=100 * tr.err[s.isin(["Pre", "Post"])].mean()))
    return pd.DataFrame(rows)

In [ ]:
rows = []
for r in STATE_READINGS:
    E = error_rates(SESS, r)
    for ctrl in ["pre", "prepost"]:
        ratio_i = E.cool.mean() / E[ctrl].mean()
        ok = E[ctrl] > 0
        ratio_ii = np.nanmedian(E.cool[ok] / E[ctrl][ok])
        p_all = mc.signrank(E.cool, E[ctrl])[0]
        per_rat = {}
        for rat, g in E.groupby("rat"):
            if len(g) >= 6:
                per_rat[rat] = mc.signrank(g.cool, g[ctrl])[0]
            else:
                tr = pd.concat([S["trials"] for S in SESS if S["rat"] == rat])
                s = tr["st_" + r]
                c = tr.err[s == "Cooling"]
                k = tr.err[s.isin(["Pre"] if ctrl == "pre" else ["Pre", "Post"])]
                per_rat[rat] = mc.ranksum(c, k)
        rows.append(dict(states=r, control=ctrl, mean_cool=E.cool.mean(), mean_ctrl=E[ctrl].mean(),
                         ratio_i=ratio_i, ratio_ii=ratio_ii, P_sessions=p_all, n=len(E),
                         **{f"P_{k}": v for k, v in per_rat.items()}))
D6 = pd.DataFrame(rows)
D6["ratio_in_tol"] = ((D6.ratio_i - 3).abs() <= 0.3) | ((D6.ratio_ii - 3).abs() <= 0.3)
D6["P<1e-5"] = D6.P_sessions < 1e-5
D6["all_rats_P<0.05"] = (D6[[c for c in D6 if c.startswith("P_MS")]] < 0.05).all(1)
print(D6.to_string(index=False))
print("rats with fewer than 6 sessions (tested over trials with rank-sum):",
      [k for k, v in SAN.groupby("rat").size().items() if v < 6])
E = error_rates(SESS, "lab")
print("per rat mean error rates (lab states):")
print(E.groupby("rat")[["pre", "cool", "post"]].mean().round(2))

**Result: partially reproduced.** Errors rise during cooling: 15.1% of cooling trials against 7.3% of pre trials, signed-rank P = 7.1e-6 over 51 sessions (9.2e-7 against pre and post), below 1e-5. The ratio is 1.9 to 2.3 under every reading, outside 2.7 to 3.3, and MS13 (5 sessions, tested over trials) does not reach P < 0.05 (P = 0.23 to 0.44); the other four rats do under the lab states.

## fig6e-errors-recover

> The error rate after cooling did not differ from the error rate before it.

In [ ]:
for r in STATE_READINGS:
    E = error_rates(SESS, r)
    p, n = mc.signrank(E.post, E.pre)
    print(f"states {r}: post - pre median {np.nanmedian(E.post - E.pre):.2f} percentage points, "
          f"mean pre {E.pre.mean():.2f}%, post {E.post.mean():.2f}%, signed-rank P = {p:.3g} (n = {n})")

**Result: reproduced.** Post error rates do not differ from pre (lab states: 6.8% against 7.3%, median difference -0.42 points, signed-rank P = 0.55, 51 sessions; P = 0.80 and 0.97 under the 12 and 10 min readings).

## fig6c-worst-trials-4-to-8

> The worst memory performance occurred between trials 4 and 8 after the onset of cooling, when MS temperature reached its minimum, together with the largest decreases of theta frequency and power.

In [ ]:
err_a = aligned(SESS, "err")
temp_a = aligned(MEAS, "temp")
f_a = {c: aligned(LFPS, c) for c in ["f_hil", "f_spec"]}
p_a = {c: aligned(LFPS, c) for c in ["p_hil_norm", "p_spec_norm"]}
w = (err_a.index >= 1) & (err_a.index <= 40)


def argext(s, fn, lo=1, hi=40):
    v = s.loc[(s.index >= lo) & (s.index <= hi)]
    return int(v.idxmax() if fn == "max" else v.idxmin())


for smooth in ["none", "5-trial mean"]:
    e = err_a["mean"] if smooth == "none" else err_a["mean"].rolling(5, center=True, min_periods=1).mean()
    k_err = argext(e, "max")
    out = dict(error_max_trial=k_err, error_max=float(e.loc[k_err]),
               temp_min_trial=argext(temp_a["mean"], "min"))
    for c, v in f_a.items():
        out[c + "_min_trial"] = argext(v["mean"], "min")
    for c, v in p_a.items():
        out[c + "_min_trial"] = argext(v["mean"], "min")
    print(f"smoothing {smooth}: {out}")
fg = {c: aligned(LFPS_GOOD, c) for c in ["f_hil", "f_spec", "p_hil_norm", "p_spec_norm"]}
print("without MS13-171130 (theta channel marked bad), minimum trial in 1 to 40:", {c: argext(v["mean"], "min") for c, v in fg.items()})
print(f"pre error incidence (aligned trials -40 to -1): {err_a.loc[err_a.index < 0, 'mean'].mean():.3f}")
print("error incidence on aligned trials 1 to 15:", err_a.loc[1:15, "mean"].round(3).tolist())
print("mean temperature on aligned trials 1 to 15:", temp_a.loc[1:15, "mean"].round(1).tolist())
print("theta frequency (Hilbert) on aligned trials 1 to 15:", f_a["f_hil"].loc[1:15, "mean"].round(2).tolist())
print("theta power (Hilbert, normalised) on aligned trials 1 to 15:", p_a["p_hil_norm"].loc[1:15, "mean"].round(3).tolist())
fig, ax = plt.subplots(4, 1, figsize=(7, 8), sharex=True)
ax[0].plot(err_a.index, err_a["mean"]); ax[0].set_ylabel("error incidence")
ax[1].plot(temp_a.index, temp_a["mean"]); ax[1].set_ylabel("MS temp (C)")
ax[2].plot(f_a["f_hil"].index, f_a["f_hil"]["mean"]); ax[2].set_ylabel("theta (Hz)")
ax[3].plot(p_a["p_hil_norm"].index, p_a["p_hil_norm"]["mean"]); ax[3].set_ylabel("theta power / pre")
for a_ in ax:
    a_.axvline(0, color="k", lw=0.5)
ax[3].set_xlabel("trial from cooling onset")
plt.tight_layout(); plt.show()

**Result: reproduced.** Without smoothing, the error incidence over aligned trials 1 to 40 peaks at trial 8 (0.31 against 0.07 before onset), and the minima of mean MS temperature (trial 7), theta frequency (trial 8, Hilbert; 6, spectral) and theta power (trial 8) all fall in trials 4 to 8. With a 5-trial mean the error peak is at trial 6. Without MS13-171130 (theta channel marked bad) the theta minima fall on the same trials (8 and 6 for frequency, 8 for power).

## fig6f-errors-temperature, fig6g-errors-theta-frequency, fig6h-errors-theta-power, fig6i-errors-not-speed

Readings (claims file, fig6f): (a) over aligned trials, error incidence against the mean of the measure across sessions, unsmoothed and with a 5-trial mean; (b) all trials pooled, the 0 or 1 error against the trial's value; (c) per rat as (a).

In [ ]:
def err_corr(SS, col, smooth=False):
    e = mc.aligned_series(SS, "err")[0]["mean"]
    v = mc.aligned_series(SS, col)[0]["mean"]
    if smooth:
        e = e.rolling(5, center=True, min_periods=1).mean()
        v = v.rolling(5, center=True, min_periods=1).mean()
    j = e.index.intersection(v.index)
    return mc.corr(e.loc[j], v.loc[j])


def err_table(SS, col):
    rows = []
    r, p, n = err_corr(SS, col)
    rows.append(dict(reading="a aligned", rat="all", R=r, P=p, n=n))
    r, p, n = err_corr(SS, col, smooth=True)
    rows.append(dict(reading="a aligned, 5-trial mean", rat="all", R=r, P=p, n=n))
    d = pd.concat([S["trials"][["err", col]] for S in SS]).dropna()
    r, p, n = mc.corr(d.err, d[col])
    rows.append(dict(reading="b pooled trials", rat="all", R=r, P=p, n=n))
    for rat in sorted(set(S["rat"] for S in SS)):
        r, p, n = err_corr([S for S in SS if S["rat"] == rat], col)
        rows.append(dict(reading="c per rat", rat=rat, R=r, P=p, n=n))
        r, p, n = err_corr([S for S in SS if S["rat"] == rat], col, smooth=True)
        rows.append(dict(reading="c per rat, 5-trial mean", rat=rat, R=r, P=p, n=n))
    return pd.DataFrame(rows)

In [ ]:
F6 = err_table(MEAS, "temp")
F6["R_in_tol"] = (F6.R + 0.35).abs() <= 0.035
print(F6.to_string(index=False))

**Result: partially reproduced.** Error incidence correlates negatively with MS temperature with P < 0.01 under the readings over aligned trials (R = -0.636 unsmoothed, -0.869 smoothed) and over pooled trials (R = -0.153, P = 4.9e-24), but no R is within -0.385 to -0.315. Per rat the unsmoothed R are -0.20 (MS12, P = 0.026), -0.28, -0.52 and -0.40.

In [ ]:
for c in ["f_hil", "f_spec"]:
    G6 = err_table(LFPS, c)
    G6["R_in_tol"] = (G6.R + 0.35).abs() <= 0.035
    print(c); print(G6.to_string(index=False))
print("restricted to the measured-temperature sessions (f_hil):")
print(err_table(MEAS_L, "f_hil").head(3).to_string(index=False))

**Result: partially reproduced.** Error incidence correlates negatively with theta frequency with P < 0.01 under every reading (aligned R = -0.707, pooled -0.116, per rat -0.30 to -0.54, all five rats P < 0.001). Only MS13 (-0.384) is within -0.385 to -0.315 and the pooled readings are not, so no reading puts R in tolerance.

In [ ]:
for c, SS in [("p_hil_norm", LFPS), ("p_hil", LFPS), ("p_spec_norm", LFPS), ("p_hil_norm without MS13-171130", LFPS_GOOD)]:
    H6 = err_table(SS, c.split(" ")[0])
    pr = H6[H6.reading == "c per rat"]
    n1 = int(((pr.R < -0.3) & (pr.P < 0.01)).sum())
    n2 = int(((pr.R > -0.3) & (pr.P < 0.01)).sum())
    print(f"{c}: rats with R < -0.3 and P < 0.01: {n1} of {len(pr)}; with R > -0.3 and P < 0.01: {n2}")
    print(pr[["rat", "R", "P", "n"]].to_string(index=False))

**Result: reproduced.** With the per-session normalised Hilbert power, 2 of 5 rats (MS13 R = -0.351, MS21 R = -0.483) have R below -0.3 with P < 0.01. The raw Hilbert power gives 3 rats and the normalised spectral power 1; no rat has R above -0.3 with P < 0.01. Without MS13-171130 (theta channel marked bad), 2 of 5 rats still count (MS13 R = -0.315, P = 4.6e-4). The count depends on the power measure: the outcome holds under the first.

In [ ]:
for c in ["speed_all", "speed_run"]:
    I6 = err_table(SESS, c)
    pr = I6[I6.reading == "c per rat"]
    print(f"{c}: rats with P < 0.05: {int((pr.P < 0.05).sum())} of {len(pr)}")
    print(I6.to_string(index=False))

**Result: not reproduced.** Two of five rats show a significant correlation of error incidence with mean trial speed over aligned trials: MS10 R = -0.276 (P = 0.0023) and MS13 R = -0.328 (P = 0.00026); the others have P 0.09 to 0.46. Pooled over sessions the aligned R is -0.146 (P = 0.11).

## suppl5j-errors-best-theta-frequency

> Error rates correlated best with theta frequency, in correlations and partial correlations, and had no significant relationship to running speed.

In [ ]:
def partial_corr(df, x, y, covars):
    import numpy.linalg as la
    d = df[[x, y] + covars].dropna()
    Z = np.column_stack([np.ones(len(d))] + [d[c].values for c in covars])
    rx = d[x].values - Z @ la.lstsq(Z, d[x].values, rcond=None)[0]
    ry = d[y].values - Z @ la.lstsq(Z, d[y].values, rcond=None)[0]
    r = np.corrcoef(rx, ry)[0, 1]
    dof = len(d) - 2 - len(covars)
    t_ = r * np.sqrt(dof / (1 - r ** 2))
    return r, 2 * stats.t.sf(abs(t_), dof), len(d)


for fc, pc, SS_name in [("f_hil", "p_hil_norm", "MEAS_L"), ("f_spec", "p_spec_norm", "MEAS_L")]:
    SS = MEAS_L
    A = pd.DataFrame({k: mc.aligned_series(SS, c)[0]["mean"] for k, c in
                      [("err", "err"), ("temp", "temp"), ("freq", fc), ("power", pc), ("speed", "speed_all")]}).dropna()
    rows = []
    for v in ["temp", "freq", "power", "speed"]:
        r, p, n = mc.corr(A.err, A[v])
        pr, pp, _ = partial_corr(A, "err", v, [o for o in ["temp", "freq", "power", "speed"] if o != v])
        rows.append(dict(measure=v, R=r, P=p, partial_R=pr, partial_P=pp, n=n))
    T5 = pd.DataFrame(rows)
    best_c = T5.measure[T5.R.abs().idxmax()]
    best_p = T5.measure[T5.partial_R.abs().idxmax()]
    sp_row = T5[T5.measure == "speed"].iloc[0]
    print(f"theta {fc}/{pc} over aligned trials of {SS_name} ({len(A)} trials):")
    print(T5.to_string(index=False))
    print(f"  largest |R|: {best_c}; largest |partial R|: {best_p}; speed P {sp_row.P:.3g}, partial P {sp_row.partial_P:.3g}")

**Result: partially reproduced.** With the Hilbert theta measures over 120 aligned trials of the measured-temperature sessions, theta frequency has the largest correlation with errors (R = -0.652) and the largest partial correlation (-0.214, P = 0.020), but speed, unrelated in the plain correlation (P = 0.78), has a significant partial correlation (0.211, P = 0.023). With the spectral measures temperature is largest in both.

## fig4e-speed-decreases-within-session

> Running speed decreased relatively monotonically within a session, over the entire session.

In [ ]:
for c in ["speed_all", "speed_run"]:
    sa = aligned(SESS, c)
    r, p, n = mc.corr(sa.index.values, sa["mean"].values, "spearman")
    per = [mc.corr(S["trials"].trial_no, S["trials"][c], "spearman")[0] for S in SESS]
    per = np.array(per)
    ps, _ = mc.signrank(per)
    print(f"{c}: aligned mean speed against trial: Spearman {r:.3f}, P = {p:.3g} (n = {n}); "
          f"per session Spearman negative in {np.sum(per < 0)} of {len(per)} (median {np.nanmedian(per):.3f}); "
          f"signed-rank of per-session values P = {ps:.3g}")
    print("   by rat, median per-session Spearman:",
          pd.Series(per, index=[S["rat"] for S in SESS]).groupby(level=0).median().round(3).to_dict())

**Result: reproduced.** The aligned mean speed falls with trial number (Spearman -0.873, P = 1.2e-38, 120 trials), the per-session Spearman correlation of speed with trial number is negative in 50 of 51 sessions (median -0.445), and the signed-rank test gives P = 7.4e-10; every rat has a negative median.

## Place fields

Pyramidal cells are the curated units typed "Pyramidal Cell" by the lab's CellExplorer file. Rate maps on the linearized maze (0 to 350 cm), left and right runs apart (Figure 3A, B), from the frames of the trials (the claims file's running reading (a); trials are the runs). Fields by the Figure 5 rule of the Methods on the map over all trials of the arm (states pooled, so that the field has the same boundaries in every state; an assumption, the paper does not say on which map fields are drawn). Figures 3 and 4 use the claims file's reading (b): the same rule with a peak above 10 Hz; reading (a), boundaries held by the dataset, is not available (the dandiset holds no place fields). Bins of 5 cm (main) and 3 cm (variant), the Methods' range. Per-state measures use the dataset's state labels (`lab`), and the 12 min reading (`m12`) as a variant.

In [ ]:
T0 = time.time()
USESS = [S for S in SESS if S["has_units"] and any(u["cell_type"] == "Pyramidal Cell" for u in S["units"])]
FIELDS, MEASURES = {}, {}
for binw in (5.0, 3.0):
    for rd in (["lab", "m12"] if binw == 5.0 else ["lab"]):
        Fs, Ms = [], []
        for S in USESS:
            F, maps, edges = mc.session_fields(S, binw=binw)
            if not len(F):
                continue
            M = mc.field_state_measures(S, F, edges, state_col="st_" + rd)
            M["peak_all"] = np.repeat(F.peak_rate.values, 3)
            M["peak_pos"] = np.repeat(F.peak_pos.values, 3)
            Fs.append(F); Ms.append(M)
        FIELDS[(binw, rd)] = pd.concat(Fs, ignore_index=True)
        MEASURES[(binw, rd)] = pd.concat(Ms, ignore_index=True)
print(f"fields computed in {time.time() - T0:.0f} s")
for k, F in FIELDS.items():
    print(f"bins {k[0]} cm, states {k[1]}: {len(F)} fields (Figure 5 rule) of {F.groupby(['session', 'u']).ngroups} cells in "
          f"{F.session.nunique()} sessions; {int((F.peak_rate > 10).sum())} with peak above 10 Hz")
print("pyramidal cells per session:", {S["session"]: sum(u["cell_type"] == "Pyramidal Cell" for u in S["units"]) for S in USESS})


def wide(key=(5.0, "lab"), fig34=True, cols=None):
    M = MEASURES[key]
    if fig34:
        M = M[M.peak_all > 10]
    W = M.pivot_table(index=["session", "rat", "u", "arm", "lo", "hi"], columns="state", values=cols, aggfunc="first")
    return W

## suppl2-place-cell-rates-decrease

> Within-field and peak firing rates of place cells decreased moderately during MS cooling (11% and 12%; P < 1e-9 and P < 1e-11).

In [ ]:
rows = []
for key in MEASURES:
    for fig34 in (True, False):
        W = wide(key, fig34, ["rate_in", "peak"])
        for m in ["rate_in", "peak"]:
            pre, cool = W[(m, "Pre")], W[(m, "Cooling")]
            ok = np.isfinite(pre) & np.isfinite(cool) & (pre > 0)
            dec = 100 * (1 - cool[ok] / pre[ok])
            rows.append(dict(bins=key[0], states=key[1], fields=("peak>10" if fig34 else "Fig5 rule"), measure=m, n=int(ok.sum()),
                             a_median_dec=np.median(dec), b_dec_of_mean=100 * (1 - cool[ok].mean() / pre[ok].mean()),
                             c_mean_dec=dec.mean(), P=mc.signrank(cool[ok], pre[ok])[0]))
R2 = pd.DataFrame(rows)
print(R2.to_string(index=False))
print("by rat (5 cm, lab, peak > 10 Hz), reading (b):")
W = wide((5.0, "lab"), True, ["rate_in", "peak"])
print(W.groupby(level="rat").apply(lambda g: pd.Series({"n": len(g), "rate_in_dec": 100 * (1 - g[("rate_in", "Cooling")].mean() / g[("rate_in", "Pre")].mean()),
                                                         "peak_dec": 100 * (1 - g[("peak", "Cooling")].mean() / g[("peak", "Pre")].mean())})).round(2))

**Result: partially reproduced.** Rates fall less than the paper reports. With 5 cm bins and fields peaking above 10 Hz, the within-field rate falls by 0.4 to 4.0% (P = 0.12) and the peak rate by 2.9 to 5.6% (P = 7.1e-4). With 3 cm bins both decreases are significant (within-field P = 0.039, peak P = 1.7e-4), but no reading brings them near 11 and 12%, and no P is below 1e-6. The mean of per-field decreases is dominated by fields with near-zero pre rates.

## suppl2ab-rates-correlated

> Firing rates of pyramidal cells before and during MS cooling were strongly correlated (R = 0.7, P < 1e-58).

In [ ]:
rows = []
for key in MEASURES:
    for fig34 in (True, False):
        W = wide(key, fig34, ["rate_in", "peak"])
        for m in ["rate_in", "peak"]:
            pre, cool = W[(m, "Pre")], W[(m, "Cooling")]
            ok = np.isfinite(pre) & np.isfinite(cool) & (pre > 0) & (cool > 0)
            r1, p1, n = mc.corr(pre[ok], cool[ok])
            r2, p2, _ = mc.corr(np.log10(pre[ok]), np.log10(cool[ok]))
            rows.append(dict(bins=key[0], states=key[1], fields=("peak>10" if fig34 else "Fig5 rule"), measure=m, n=n,
                             R_linear=r1, P_linear=p1, R_log=r2, P_log=p2))
AB = pd.DataFrame(rows)
AB["in_tol"] = ((AB.R_linear - 0.7).abs() <= 0.07) & (AB.P_linear < 1e-20) | ((AB.R_log - 0.7).abs() <= 0.07) & (AB.P_log < 1e-20)
print(AB.to_string(index=False))

**Result: reproduced.** The within-field rates of 1185 fields before and during cooling correlate with R = 0.658 (P = 1.2e-147), within 0.63 to 0.77; peak rates give 0.619 and log scales 0.49 to 0.57.

## fig4b-fp-flfp-decrease

> MS cooling decreased both the oscillation frequency of place cells (fp) and the LFP theta frequency in their fields (fLFP) by about the same degree, and both recovered (fLFP 8.4, 7.7, 8.5 Hz; fp 10.6, 9.5, 10.4 Hz before, during and after; n = 365).

Fields are kept when the autocorrelogram has a peak between 50 and 150 ms in each state (criterion). fLFP readings: (a) at the field's spikes, (b) over the time in the field (from the theta cycles whose middle falls in the field).

In [ ]:
EXP = dict(flfp=(8.4, 7.7, 8.5), flfp_tol=(0.38, 0.36, 0.29), fp=(10.6, 9.5, 10.4), fp_tol=(0.95, 0.84, 0.72))


def fp_fields(key=(5.0, "lab"), fig34=True):
    W = wide(key, fig34, ["fp", "flfp_spk", "flfp_time", "size", "v_time", "v_spk", "info", "prec_slope", "rate_in", "peak"])
    ok = W["fp"].notna().all(1) & W["flfp_spk"].notna().all(1)
    return W[ok]


rows = []
for key in MEASURES:
    for fig34 in (True, False):
        W = fp_fields(key, fig34)
        for fl in ["flfp_spk", "flfp_time"]:
            m_fp = [W[("fp", s)].mean() for s in mc.STATES]
            m_fl = [W[(fl, s)].mean() for s in mc.STATES]
            inside = all(abs(m_fl[i] - EXP["flfp"][i]) <= EXP["flfp_tol"][i] for i in range(3)) and \
                all(abs(m_fp[i] - EXP["fp"][i]) <= EXP["fp_tol"][i] for i in range(3))
            rows.append(dict(bins=key[0], states=key[1], fields=("peak>10" if fig34 else "Fig5 rule"), flfp=fl, n=len(W),
                             fp_pre=m_fp[0], fp_cool=m_fp[1], fp_post=m_fp[2],
                             flfp_pre=m_fl[0], flfp_cool=m_fl[1], flfp_post=m_fl[2],
                             P_fp=mc.signrank(W[("fp", "Cooling")], W[("fp", "Pre")])[0],
                             P_flfp=mc.signrank(W[(fl, "Cooling")], W[(fl, "Pre")])[0],
                             all_six_in_tol=inside))
B4 = pd.DataFrame(rows)
print(B4.to_string(index=False))
W = fp_fields()
print(f"main (5 cm, lab, peak > 10 Hz): {len(W)} fields of {len(wide((5.0, 'lab'), True, ['fp']))} kept; SD over fields "
      f"fp {[round(float(W[('fp', s)].std()), 2) for s in mc.STATES]}, fLFP (spikes) {[round(float(W[('flfp_spk', s)].std()), 2) for s in mc.STATES]}")
print("by rat, mean fp and fLFP (spikes) per state:")
print(W.groupby(level="rat")[[("fp", s) for s in mc.STATES] + [("flfp_spk", s) for s in mc.STATES]].mean().round(2))
print("fields per rat:", W.groupby(level="rat").size().to_dict())

**Result: reproduced.** Over 878 fields with an autocorrelogram peak in every state (5 cm bins, fields above 10 Hz), fp is 10.50, 9.64 and 10.36 Hz and fLFP at the spikes 8.27, 7.62 and 8.43 Hz before, during and after cooling, all six within tolerance, with both decreases significant (signed-rank P = 1.9e-75 and 1.6e-101). Every other reading also puts all six within tolerance. The paper has 365 fields.

## fig4c-fp-flfp-correlation

> fp and fLFP were correlated across place fields before, during and after cooling (R = 0.61, 0.59 and 0.36; P < 1e-6 each), with the range of fp larger than that of fLFP.

In [ ]:
rows = []
for key in MEASURES:
    for fig34 in (True, False):
        W = fp_fields(key, fig34)
        for fl in ["flfp_spk", "flfp_time"]:
            r = {}
            for s in mc.STATES:
                R_, P_, n = mc.corr(W[("fp", s)], W[(fl, s)])
                r[f"R_{s}"], r[f"P_{s}"] = R_, P_
            sd = [W[("fp", s)].std() > W[(fl, s)].std() for s in ("Pre", "Cooling")]
            rows.append(dict(bins=key[0], states=key[1], fields=("peak>10" if fig34 else "Fig5 rule"), flfp=fl, n=len(W), **r,
                             sd_fp_gt_flfp_pre_cool=all(sd)))
C4 = pd.DataFrame(rows)
C4["all_in_tol"] = ((C4.R_Pre - 0.61).abs() <= 0.061) & ((C4.R_Cooling - 0.59).abs() <= 0.059) & ((C4.R_Post - 0.36).abs() <= 0.036)
print(C4.to_string(index=False))

**Result: partially reproduced.** fp and fLFP correlate positively in every state with P below 1e-11 under every reading, and fp spreads more than fLFP (SD 1.21 against 0.67 Hz pre, 1.03 against 0.53 Hz cooling), but R is lower than the paper's: 0.25, 0.43 and 0.24 at the spikes and 0.36, 0.44 and 0.31 over the time in the field, against 0.61, 0.59 and 0.36.

## fig4ac-fp-flfp-difference-preserved

> During MS cooling fp and fLFP decreased in parallel, so the difference fp - fLFP remained constant.

In [ ]:
for key in [(5.0, "lab"), (3.0, "lab"), (5.0, "m12")]:
    W = fp_fields(key)
    for fl in ["flfp_spk", "flfp_time"]:
        d = {s: W[("fp", s)] - W[(fl, s)] for s in mc.STATES}
        p, n = mc.signrank(d["Cooling"], d["Pre"])
        print(f"bins {key[0]}, states {key[1]}, fLFP {fl}: fp - fLFP mean pre {d['Pre'].mean():.2f}, cooling {d['Cooling'].mean():.2f}, "
              f"post {d['Post'].mean():.2f} Hz; median change cooling - pre {np.median(d['Cooling'] - d['Pre']):.3f} Hz; signed-rank P = {p:.3g} (n = {n})")
print("paper's means give fp - fLFP of 2.2, 1.8 and 1.9 Hz")

**Result: not reproduced.** fp - fLFP falls slightly during cooling (2.22 to 2.02 Hz, median change -0.14 Hz), and the signed-rank test over 878 fields finds it (P = 1.1e-6; P = 2.0e-4 with fLFP over the time in the field). The paper's own means give 2.2 and 1.8 Hz.

## fig3c-field-size-correlated, fig3d-size-fp-negative, fig3e-model-field-size, fig3f-information-unchanged

In [ ]:
for key in MEASURES:
    W = wide(key, True, ["size"])
    ok = W[("size", "Pre")].notna() & W[("size", "Cooling")].notna()
    r, p, n = mc.corr(W.loc[ok, ("size", "Pre")], W.loc[ok, ("size", "Cooling")])
    dd = W.loc[ok, ("size", "Cooling")] - W.loc[ok, ("size", "Pre")]
    print(f"bins {key[0]}, states {key[1]}: R of size pre against cooling {r:.3f} (P = {p:.2g}, n = {n}); "
          f"median size pre {W.loc[ok, ('size', 'Pre')].median():.1f} cm, cooling {W.loc[ok, ('size', 'Cooling')].median():.1f} cm; "
          f"median change {dd.median():.1f} cm, signed-rank P = {mc.signrank(dd)[0]:.3g}")

**Result: not reproduced.** Field size before and during cooling correlates with R = 0.465 (1193 fields, 5 cm bins), 0.490 (12 min states) and 0.410 (3 cm bins), all below 0.5. The median size does not change (60 cm in both; signed-rank P = 0.082).

In [ ]:
for key in MEASURES:
    W = fp_fields(key)
    out = []
    for s in mc.STATES:
        r, p, n = mc.corr(W[("size", s)], W[("fp", s)])
        lr = stats.linregress(W[("fp", s)], W[("size", s)])
        out.append(f"{s}: R {r:.3f} (P = {p:.2g}), slope {lr.slope:.2f} cm/Hz, intercept {lr.intercept:.1f} cm")
    print(f"bins {key[0]}, states {key[1]}, n = {len(W)}: " + "; ".join(out))

**Result: partially reproduced.** Field size and fp correlate negatively in every state (R = -0.181, -0.204 and -0.141; P = 7e-8, 1.1e-9 and 2.8e-5), weaker than -0.49 and -0.40 and with P above 1e-9 in two states. The regression slopes are -7.6, -9.9 and -7.1 cm/Hz; the cooling intercept is higher, not lower.

In [ ]:
for key in MEASURES:
    W = fp_fields(key)
    for v in ["v_time", "v_spk"]:
        for fl in ["flfp_spk", "flfp_time"]:
            out = []
            for s in mc.STATES:
                dfp = W[("fp", s)] - W[(fl, s)]
                ok = dfp > 0
                pred = W[(v, s)][ok] / dfp[ok]
                r, p, n = mc.corr(W[("size", s)][ok], pred)
                out.append(f"{s} R {r:.3f} (P = {p:.2g}, n = {n}, left out {int((~ok).sum())})")
            print(f"bins {key[0]}, states {key[1]}, v {v}, fLFP {fl}: " + "; ".join(out))

**Result: not reproduced.** The size predicted by v / (fp - fLFP) does not correlate with the measured size before cooling under any reading (R -0.032 to 0.059, P >= 0.088), and the cooling and post R are 0.134 at most.

In [ ]:
for key in MEASURES:
    for fig34 in (True, False):
        W = wide(key, fig34, ["info"]).groupby(level=["session", "rat", "u", "arm"]).first()
        out = []
        for a_, b_ in [("Pre", "Cooling"), ("Cooling", "Post"), ("Pre", "Post")]:
            p, n = mc.signrank(W[("info", b_)], W[("info", a_)])
            out.append(f"{a_}-{b_}: median change {np.nanmedian(W[('info', b_)] - W[('info', a_)]):.3f} bits/spike, P = {p:.3g}")
        print(f"bins {key[0]}, states {key[1]}, {'peak>10' if fig34 else 'Fig5 rule'} (n = {len(W)}): " + "; ".join(out))
print("information is per map (cell and arm), counted once per map with a field; a cell with fields on both arms counts once per arm")
# equal trials per state: Skaggs information is biased upward with fewer data, and the states differ in trial counts
rng_info = np.random.default_rng(31)
T0 = time.time()
IEQ = []
for S in USESS:
    F = FIELDS[(5.0, "lab")]
    F = F[F.session == S["session"]]
    if len(F):
        I = mc.info_equal_trials(S, F, np.arange(0, 355, 5.0), "st_lab", n_draws=20, rng=rng_info)
        I["peak10"] = [bool(((F.u == u) & (F.arm == a) & (F.peak_rate > 10)).any()) for u, a in zip(I.u, I.arm)]
        IEQ.append(I)
IEQ = pd.concat(IEQ, ignore_index=True)
print(f"equal-trial information ({time.time() - T0:.0f} s): trials per state and arm, median {IEQ.trials.median():.0f} (range {IEQ.trials.min()} to {IEQ.trials.max()})")
tc = pd.concat([S["trials"].assign(session=S["session"]) for S in USESS]).groupby(["session", "arm", "st_lab"]).size().unstack()
print("trials per state and arm (unequal reading), median:", tc.median().to_dict())
for lab_, I in [("peak>10", IEQ[IEQ.peak10]), ("Fig5 rule", IEQ)]:
    out = []
    for a_, b_ in [("Pre", "Cooling"), ("Cooling", "Post"), ("Pre", "Post")]:
        p, n = mc.signrank(I[b_], I[a_])
        out.append(f"{a_}-{b_}: median change {np.nanmedian(I[b_] - I[a_]):.3f} bits/spike, P = {p:.3g}")
    print(f"equal trials, bins 5.0, states lab, {lab_} (n = {len(I)}; median pre {I.Pre.median():.3f}, cooling {I.Cooling.median():.3f}, post {I.Post.median():.3f}): " + "; ".join(out))

**Result: not reproduced.** Spatial information does not change from pre to cooling (median change -0.015 bits per spike, signed-rank P = 0.35 over 908 maps) or from cooling to post (P = 0.078), but it falls from pre to post (median change -0.038, P = 0.0054), so one of the three comparisons is significant. Every field set and bin size finds the pre to post decrease (P = 4e-6 to 0.0054). With the trials of each state subsampled to the same number (median 17 per state and arm, 20 draws), which removes the bias of unequal state sizes, pre to post still falls (median change -0.030 bits per spike, P = 0.016 over 902 maps; P = 0.0047 with the Figure 5 rule fields), while pre to cooling (P = 0.22) and cooling to post (P = 0.22) do not differ.

## fig4d-precession-slope-decrease

> The phase precession slope decreased slightly but significantly during MS cooling (P < 0.007 and P < 0.0003).

Circular-linear regression (Kempter et al. 2012) of theta phase on position within the field, position scaled to the field (0 to 1), so the slope is in radians per field; fields with at least 10 spikes in a state (standard choice). The slope is searched over plus and minus 2 cycles per field (4 pi rad; an assumption, as the paper gives no range); 1 cycle, which pins many fields at its bound, and 3 cycles are variants. The fraction of slopes at the bound is printed.

In [ ]:
BOUND = {"prec_slope": 4 * np.pi, "prec_slope_1c": 2 * np.pi, "prec_slope_3c": 6 * np.pi}
for col, rng_name in [("prec_slope", "2 cycles (main)"), ("prec_slope_1c", "1 cycle"), ("prec_slope_3c", "3 cycles")]:
    for key in MEASURES:
        W = wide(key, True, [col])
        W = W.dropna()
        s = {k: W[(col, k)] for k in mc.STATES}
        at_b = {k: float((np.abs(np.abs(v) - BOUND[col]) < 1e-3).mean()) for k, v in s.items()}
        out = [f"n = {len(W)}; median slope pre {s['Pre'].median():.3f}, cooling {s['Cooling'].median():.3f}, post {s['Post'].median():.3f} rad/field; "
               f"fraction at the search bound pre {at_b['Pre']:.3f}, cooling {at_b['Cooling']:.3f}, post {at_b['Post']:.3f}"]
        for a_ in ["Pre", "Post"]:
            p_mag, _ = mc.signrank(s["Cooling"].abs(), s[a_].abs())
            p_sig, _ = mc.signrank(s["Cooling"], s[a_])
            out.append(f"cooling vs {a_}: |slope| median change {np.median(s['Cooling'].abs() - s[a_].abs()):.4f} (P = {p_mag:.3g}); "
                       f"signed slope median change {np.median(s['Cooling'] - s[a_]):.4f} (P = {p_sig:.3g})")
        print(f"range {rng_name}, bins {key[0]}, states {key[1]}: " + "; ".join(out))
    if col == "prec_slope":
        Wb = wide((5.0, "lab"), True, [col]).dropna()
        off = (np.abs(np.abs(Wb) - BOUND[col]) >= 1e-3).all(1)
        Wb = Wb[off]
        print(f"   main reading without fields at the bound in any state (n = {len(Wb)}): cooling vs pre |slope| P = "
              f"{mc.signrank(Wb[(col, 'Cooling')].abs(), Wb[(col, 'Pre')].abs())[0]:.3g}, signed P = {mc.signrank(Wb[(col, 'Cooling')], Wb[(col, 'Pre')])[0]:.3g}; "
              f"cooling vs post |slope| P = {mc.signrank(Wb[(col, 'Cooling')].abs(), Wb[(col, 'Post')].abs())[0]:.3g}, signed P = {mc.signrank(Wb[(col, 'Cooling')], Wb[(col, 'Post')])[0]:.3g}")
W = wide((5.0, "lab"), True, ["prec_slope"]).dropna()
print(f"fraction of negative slopes (pre, main reading): {(W[('prec_slope', 'Pre')] < 0).mean():.3f}")


**Result: partially reproduced (assumed).** With the slope searched over plus and minus 2 cycles per field (main reading; 8 to 10% of the slopes at the bound), precession is flatter during cooling than before (median change of |slope| -0.13 rad per field, signed-rank P = 0.016 over 1044 fields) but not significantly flatter than after (P = 0.073); the signed slope gives P = 0.065 and 0.069. With 3 cm bins the magnitude reading gives P = 0.0041 against pre and 0.083 against post, and the signed reading 0.056 and 0.0062. No reading gives P < 0.05 for both comparisons. With a range of 1 cycle, 23 to 32% of the slopes sit at the bound and no comparison is significant (P 0.054 to 0.94); with 3 cycles at most one comparison per reading is significant. The outcome rests on the slope range, which the paper does not give.

## suppl3e-precession-temperature, fig4a-frequency-temperature-slopes

Per field and trial (reading a): slope and fp from that trial's in-field spikes, with the mean MS temperature at those spikes. Per field and state (reading b): the state's values with the state's mean temperature at the field's spikes. Measured-temperature sessions only. For a per-trial slope at least 5 spikes are needed (standard choice).

In [ ]:
T0 = time.time()
MEAS_NAMES = {S["session"] for S in MEAS}
F5 = FIELDS[(5.0, "lab")]
F5 = F5[F5.peak_rate > 10]
rows = []
for S in USESS:
    if S["session"] not in MEAS_NAMES:
        continue
    sp = S["sp"]
    tt, T = S["temp"]
    ok = np.isfinite(T)
    sp_temp = np.interp(sp.t, tt[ok], T[ok])
    for _, f in F5[F5.session == S["session"]].iterrows():
        m = (sp.u.values == f.u) & (sp.arm.values == f.arm) & (sp.lin.values >= f.lo) & (sp.lin.values < f.hi)
        su = sp[m].assign(temp=sp_temp[m])
        for ti, g in su.groupby("trial"):
            r = dict(session=S["session"], rat=S["rat"], u=f.u, arm=f.arm, lo=f.lo, trial=ti, n=len(g),
                     state=S["trials"].st_lab.values[ti], temp=g.temp.mean(), flfp=g.theta_f.mean())
            if len(g) >= 5:
                x = (g.lin.values - f.lo) / (f.hi - f.lo)
                ph = np.mod(g.phase_u.values, 2 * np.pi)
                r["slope"] = mc.circ_lin_slope(x, ph)[0]
                r["slope_1c"] = mc.circ_lin_slope(x, ph, 1.0)[0]
            r["fp"] = mc.acg_frequency([g.t.values * 1000])[0] if len(g) >= 2 else np.nan
            rows.append(r)
FT = pd.DataFrame(rows)
print(f"field-trial values: {len(FT)} passes of {FT.groupby(['session', 'u', 'arm', 'lo']).ngroups} fields in {FT.session.nunique()} sessions; "
      f"with a slope {FT.slope.notna().sum()}, with an fp {FT.fp.notna().sum()} ({time.time() - T0:.0f} s)")
# per field and state: the state's mean temperature at the field's spikes
FS = FT.groupby(["session", "rat", "u", "arm", "lo", "state"]).apply(
    lambda g: pd.Series(dict(temp=np.average(g.temp, weights=g.n), n=g.n.sum()))).reset_index()
MW = MEASURES[(5.0, "lab")]
MW = MW[MW.peak_all > 10]
FS = FS.merge(MW[["session", "u", "arm", "lo", "state", "prec_slope", "prec_slope_1c", "fp", "flfp_spk", "flfp_time"]],
              on=["session", "u", "arm", "lo", "state"])
print(f"field-state values: {len(FS)}")

In [ ]:
for sc, fc, bnd, lab_ in [("slope", "prec_slope", 4 * np.pi, "2 cycles (main)"), ("slope_1c", "prec_slope_1c", 2 * np.pi, "1 cycle")]:
    d = FT.dropna(subset=[sc, "temp"])
    r, p, n = mc.corr(d.temp, d[sc])
    print(f"range {lab_}: (a) field and trial: R = {r:.4f}, P = {p:.3g}, n = {n}; fraction at the bound {(np.abs(np.abs(d[sc]) - bnd) < 1e-3).mean():.3f}")
    d = FS.dropna(subset=[fc, "temp"])
    r2, p2, n2 = mc.corr(d.temp, d[fc])
    print(f"range {lab_}: (b) field and state: R = {r2:.4f}, P = {p2:.3g}, n = {n2}")
    print(f"   within tolerance (-0.0297 to -0.0243) with P < 1e-10: (a) {abs(r + 0.027) <= 0.0027 and p < 1e-10}, (b) {abs(r2 + 0.027) <= 0.0027 and p2 < 1e-10}")


**Result: partially reproduced.** Per field and trial (31617 values; range of 2 cycles per field, 18% of the per-trial slopes at the bound), the precession slope correlates with MS temperature with R = -0.0262, within -0.0297 to -0.0243, but P = 3.2e-6, not below 1e-10. Per field and state, R = -0.0274 with P = 0.22. With a range of 1 cycle the values are R = -0.0273 (P = 1.2e-6) and -0.0243 (P = 0.28).

In [ ]:
rows = []
for reading, D, fpc, flc in [("a field-trial", FT, "fp", "flfp"), ("b field-state", FS, "fp", "flfp_spk"), ("b field-state, fLFP over time", FS, "fp", "flfp_time")]:
    for meas, col in [("fLFP", flc), ("fp", fpc)]:
        dd = D.dropna(subset=["temp", col])
        lr = stats.linregress(dd.temp, dd[col])
        rows.append(dict(reading=reading, measure=meas, R=lr.rvalue, slope_Hz_per_C=lr.slope, P=lr.pvalue, n=len(dd)))
A4 = pd.DataFrame(rows)
print(A4.to_string(index=False))
print("expected: fLFP R 0.66 (0.594 to 0.726), slope 0.071 (0.0639 to 0.0781); fp R 0.38 (0.342 to 0.418), slope 0.09 (0.081 to 0.099)")
print(f"field-trial fp: passes with fewer than 2 spikes or no autocorrelogram peak are dropped ({FT.fp.isna().sum()} of {len(FT)})")

**Result: partially reproduced.** Both rise with MS temperature at P below 1e-30. Per field and state with fLFP over the time in the field, fLFP gives R = 0.685 and a slope of 0.068 Hz per C, both within tolerance, but fp gives R = 0.301 and 0.067 Hz per C, both below. Per field and trial the R values are lower (0.18 and 0.16).

## fig4f-speed-theta-power

> Running speed correlated with theta power.

In [ ]:
d = pd.concat([S["trials"][["speed_all", "p_hil", "p_hil_norm", "p_spec_norm"]].assign(rat=S["rat"]) for S in LFPS]).dropna()
for c in ["p_hil", "p_hil_norm", "p_spec_norm"]:
    r, p, n = mc.corr(d.speed_all, d[c])
    per = {rat: mc.corr(g.speed_all, g[c])[:2] for rat, g in d.groupby("rat")}
    print(f"{c}: (a) all trials R = {r:.3f}, P = {p:.3g}, n = {n}; (b) per rat " +
          ", ".join(f"{k} R {v[0]:.3f} P {v[1]:.2g}" for k, v in per.items()) +
          f"; rats with P < 0.05: {sum(v[1] < 0.05 for v in per.values())} of {len(per)}")

**Result: reproduced.** Over all 7087 trials, mean speed correlates positively with theta power (R = 0.071, P = 2.7e-9, raw Hilbert power). Per rat 3 to 4 of 5 rats reach P < 0.05 depending on the power measure; MS10 never does.

## fig4gh-fp-speed

> fp depended on running speed more strongly than fLFP, so fp - fLFP increased with speed, and fp during cooling was lower at all velocities.

In [ ]:
T0 = time.time()
rng = np.random.default_rng(4)


def speed_bins(binv):
    rows = []
    for S in USESS:
        if not S["has_lfp"]:
            continue
        sp = S["sp"]
        st = S["trials"].st_lab.values
        for _, f in F5[F5.session == S["session"]].iterrows():
            m = (sp.u.values == f.u) & (sp.arm.values == f.arm) & (sp.lin.values >= f.lo) & (sp.lin.values < f.hi)
            su = sp[m].assign(vb=np.floor(sp.speed.values[m] / binv) * binv, state=st[sp.trial.values[m]])
            for (vb, s), g in list(su.groupby(["vb", "state"])) + [((vb, "all"), g) for vb, g in su.groupby("vb")]:
                if s not in ("Pre", "Cooling", "all") or len(g) < 2:
                    continue
                fpv = mc.acg_frequency([gg.t.values * 1000 for _, gg in g.groupby("trial")])[0]
                rows.append(dict(session=S["session"], u=f.u, arm=f.arm, lo=f.lo, vb=vb, state=s,
                                 fp=fpv, flfp=g.theta_f.mean(), n=len(g)))
    return pd.DataFrame(rows)


for binv in (10.0, 20.0):
    SB = speed_bins(binv)
    A = SB[(SB.state == "all")].dropna(subset=["fp"])
    A = A[A.vb >= 10]
    l1 = stats.linregress(A.vb + binv / 2, A.fp)
    l2 = stats.linregress(A.vb + binv / 2, A.flfp)
    keys = A.groupby(["session", "u", "arm", "lo"]).ngroup().values
    boots = []
    for _ in range(500):
        pick = rng.integers(0, keys.max() + 1, keys.max() + 1)
        idx = np.concatenate([np.where(keys == k)[0] for k in pick])
        x = A.vb.values[idx]
        boots.append(np.polyfit(x, A.fp.values[idx], 1)[0] - np.polyfit(x, A.flfp.values[idx], 1)[0])
    lo_, hi_ = np.percentile(boots, [2.5, 97.5])
    print(f"bins {binv:.0f} cm/s, states pooled, {len(A)} field-bins of {len(np.unique(keys))} fields: fp slope {l1.slope:.4f} Hz per cm/s "
          f"(P = {l1.pvalue:.3g}), fLFP slope {l2.slope:.4f} (P = {l2.pvalue:.3g}); difference 95% bootstrap interval [{lo_:.4f}, {hi_:.4f}]")
    P_ = SB[SB.state.isin(["Pre", "Cooling"])].dropna(subset=["fp"]).pivot_table(index=["session", "u", "arm", "lo", "vb"], columns="state", values="fp")
    P_ = P_.dropna()
    g = P_.groupby(level="vb").agg(n=("Pre", "size"), pre=("Pre", "median"), cool=("Cooling", "median"))
    g = g[g.n >= 10]
    g["cool_lower"] = g.cool < g.pre
    print(g.round(3).to_string())
    print(f"  speed bins with at least 10 fields in both states: {len(g)}, cooling lower in {int(g.cool_lower.sum())}")
print(f"({time.time() - T0:.0f} s)")

**Result: partially reproduced.** In 10 cm/s bins with states pooled, fp rises with speed (0.0106 Hz per cm/s, P = 2.7e-60) faster than fLFP (0.0045; bootstrap interval of the difference 0.0044 to 0.0076), but fp during cooling is below pre in 11 of 14 speed bins, not all (0, 30 and 120 cm/s are not). With 20 cm/s bins it is below in all 8.

## suppl2e-trial-speed-theta-frequency

> Over trials, mean theta frequency correlated positively with mean running speed before, during and after MS cooling, with the largest changes at 10 to 40 cm/s.

In [ ]:
d = pd.concat([S["trials"][["speed_all", "f_hil", "f_spec", "st_lab"]] for S in LFPS])
for fc in ["f_hil", "f_spec"]:
    out = []
    nst = 0
    for s in mc.STATES:
        g = d[d.st_lab == s].dropna(subset=[fc, "speed_all"])
        r, p, n = mc.corr(g.speed_all, g[fc])
        lo_ = g[(g.speed_all >= 10) & (g.speed_all < 40)]
        hi_ = g[g.speed_all >= 40]
        s1 = stats.linregress(lo_.speed_all, lo_[fc]).slope if len(lo_) > 2 else np.nan
        s2 = stats.linregress(hi_.speed_all, hi_[fc]).slope
        nst += s1 > s2
        out.append(f"{s}: R {r:.3f} (P = {p:.3g}, n = {n}); slope 10-40 cm/s {s1:.4f} (n = {len(lo_)}), above 40 {s2:.4f} Hz per cm/s")
    print(f"{fc}: " + "; ".join(out) + f"; states with the low-speed slope larger: {nst}")
print(f"trials with mean speed below 40 cm/s: {(d.speed_all < 40).sum()} of {len(d)}")

**Result: reproduced.** Trial theta frequency correlates positively with trial speed in pre, cooling and post (Hilbert R = 0.360, 0.259 and 0.387; P below 1e-33), and the slope over trials of 10 to 40 cm/s exceeds that above 40 cm/s in cooling and post (2 of 3 states; 3 of 3 with the spectral reading). Only 61 of 7087 trials have a mean speed below 40 cm/s.

## suppl3-theta-cycle-duration

> Theta cycles were consistently longer during MS cooling at every position on the track and returned to control duration after cooling, and the duration profile did not correlate with the speed profile along the track.

In [ ]:
rows = []
C = []
for S in LFPS:
    c = S["cycles"].copy()
    c["state"] = S["trials"].st_lab.values[c.trial.values]
    c["session"] = S["session"]
    C.append(c)
    g = c.groupby("state").dur.median()
    if {"Pre", "Cooling", "Post"} <= set(g.index):
        rows.append(dict(session=S["session"], pre=g["Pre"], cool=g["Cooling"], post=g["Post"]))
CD = pd.DataFrame(rows)
C = pd.concat(C)
print(f"sessions {len(CD)}; median cycle duration (ms) pre {1000 * CD.pre.median():.1f}, cooling {1000 * CD.cool.median():.1f}, post {1000 * CD.post.median():.1f}")
print(f"cooling vs pre P = {mc.signrank(CD.cool, CD.pre)[0]:.3g} (longer in {(CD.cool > CD.pre).sum()}); cooling vs post P = {mc.signrank(CD.cool, CD.post)[0]:.3g} "
      f"(longer in {(CD.cool > CD.post).sum()}); post vs pre P = {mc.signrank(CD.post, CD.pre)[0]:.3g}, median post - pre {1000 * np.median(CD.post - CD.pre):.2f} ms")
for arm in ["Left", "Right"]:
    cc = C[(C.arm == arm) & (C.lin >= 0)]
    cc = cc.assign(pb=np.floor(cc.lin / 5) * 5)
    g = cc.groupby(["pb", "state"]).agg(dur=("dur", "mean"), speed=("speed", "mean"), n=("dur", "size")).unstack("state")
    g = g[(g[("n", "Pre")] >= 50) & (g[("n", "Cooling")] >= 50)]
    longer = (g[("dur", "Cooling")] > g[("dur", "Pre")]).mean()
    r, p, n = mc.corr(g[("dur", "Pre")], g[("speed", "Pre")])
    print(f"{arm} runs: {len(g)} position bins (at least 50 cycles per state); cooling longer than pre in {100 * longer:.1f}% of bins; "
          f"pre duration against pre speed over bins R = {r:.3f}, P = {p:.3g}")

**Result: partially reproduced.** Theta cycles are longer during cooling (median 130.4 against 117.6 ms pre and post; P = 1.2e-9 and 1.1e-9 over 50 sessions), post does not differ from pre (P = 0.96), and cooling is longer in every position bin of both arms. The pre duration profile, however, correlates with the speed profile along the track (R = -0.43 and -0.32; P = 0.0047 and 0.0089).

## disc-interneuron-rates-decrease

> Firing rates of interneurons decreased during MS cooling by about 14%, about the same degree as place cells.

Rate of a unit in a state: its spikes in the state's trials over the summed duration of those trials (running reading a), or its spikes above 10 cm/s over the time above 10 cm/s in those trials (reading b).

In [ ]:
def unit_state_rates(S, state_col="st_lab"):
    tr = S["trials"]
    b = S["beh"]
    bti = mc._beh_trial_index(S)
    dt = np.gradient(b["t"])
    st_tr = tr[state_col].values
    sp = S["sp"]
    st_sp = st_tr[sp.trial.values]
    run_sp = sp.speed.values > mc.RUN_SPEED
    rows = []
    dur = {s: (tr.stop - tr.start)[st_tr == s].sum() for s in mc.STATES}
    run_t = {}
    for s in mc.STATES:
        m = bti >= 0
        m[m] = st_tr[bti[m]] == s
        run_t[s] = dt[m & (np.nan_to_num(b["speed"]) > mc.RUN_SPEED)].sum()
    for k, U in enumerate(S["units"]):
        r = dict(session=S["session"], rat=S["rat"], u=k, cell_type=U["cell_type"])
        mu = sp.u.values == k
        for s in mc.STATES:
            r[f"all_{s}"] = (mu & (st_sp == s)).sum() / dur[s] if dur[s] > 0 else np.nan
            r[f"run_{s}"] = (mu & (st_sp == s) & run_sp).sum() / run_t[s] if run_t[s] > 0 else np.nan
        rows.append(r)
    return pd.DataFrame(rows)


UR = pd.concat([unit_state_rates(S) for S in SESS if S["has_units"] and len(S["units"])], ignore_index=True)
print("curated units by type:", UR.cell_type.replace("", "(none)").value_counts().to_dict())

In [ ]:
rows = []
for lab_, types in [("a narrow", ["Narrow Interneuron"]), ("b narrow and wide", ["Narrow Interneuron", "Wide Interneuron"]),
                    ("pyramidal (reference)", ["Pyramidal Cell"])]:
    g = UR[UR.cell_type.isin(types)]
    for run in ["all", "run"]:
        pre, cool = g[f"{run}_Pre"], g[f"{run}_Cooling"]
        ok = (pre > 0) & np.isfinite(cool)
        dec = 100 * (1 - cool[ok] / pre[ok])
        rows.append(dict(units=lab_, running=run, n=int(ok.sum()), a_median_dec=np.median(dec),
                         b_dec_of_mean=100 * (1 - cool[ok].mean() / pre[ok].mean()), c_mean_dec=dec.mean(),
                         P=mc.signrank(cool[ok], pre[ok])[0]))
IR = pd.DataFrame(rows)
IR["in_12.6_15.4"] = IR[["a_median_dec", "b_dec_of_mean", "c_mean_dec"]].apply(lambda r: bool(((r - 14).abs() <= 1.4).any()), axis=1)
print(IR.to_string(index=False))
g = UR[UR.cell_type.isin(["Narrow Interneuron", "Wide Interneuron"])]
print("interneurons by rat (n, decrease of the mean, all frames):")
print(g.groupby("rat").apply(lambda x: pd.Series(dict(n=len(x), dec=100 * (1 - x.all_Cooling.mean() / x.all_Pre.mean())))).round(2))

**Result: partially reproduced.** Narrow waveform interneurons fire less during cooling (194 units, signed-rank P = 0.023), but by 4.2% (median of per-unit decreases) or 8.7% (decrease of the mean), not 14%; the mean of per-unit decreases is negative, pulled by units with near-zero pre rates; narrow and wide together fall by 3.1 to 8.0% (P = 0.11). Pyramidal cells fall by 6.1 to 7.1% (P = 5e-4).

## suppl5gh-rates-follow-temperature

> Firing rates of pyramidal cells and putative interneurons decreased immediately after MS cooling onset and their changes correlated with MS temperature, not with the delayed hippocampal temperature.

In [ ]:
def trial_rates(S, types):
    tr = S["trials"]
    sp = S["sp"]
    dur = (tr.stop - tr.start).values
    out = []
    for k, U in enumerate(S["units"]):
        if U["cell_type"] not in types:
            continue
        cnt = np.bincount(sp.trial.values[sp.u.values == k], minlength=len(tr))
        rate = cnt / dur
        pre = rate[tr.st_lab.values == "Pre"].mean()
        keep = np.ones(len(tr), bool)
        if len(S["onsets"]) > 1:
            keep &= tr.start.values < S["onsets"][1]
        out.append(pd.DataFrame(dict(session=S["session"], u=k, aligned=tr.aligned.values, rate=rate,
                                     norm=rate / pre if pre > 0 else np.nan, temp=tr.temp.values))[keep])
    return pd.concat(out) if out else pd.DataFrame()


for cls, types in [("interneurons", ["Narrow Interneuron", "Wide Interneuron"]), ("pyramidal cells", ["Pyramidal Cell"])]:
    TRt = pd.concat([trial_rates(S, types) for S in SESS if S["has_units"]])
    TRt = TRt[(TRt.aligned >= -40) & (TRt.aligned <= 80)]
    n_units = TRt.groupby(["session", "u"]).ngroups
    n_sess = TRt.session.nunique()
    for col in ["norm", "rate"]:
        per = TRt.groupby(["session", "u"]).apply(lambda g: pd.Series(dict(
            early=g[(g.aligned >= 1) & (g.aligned <= 5)][col].mean(), pre=g[g.aligned < 0][col].mean())))
        p_early, n_ = mc.signrank(per.early, per.pre)
        dd = (per.early - per.pre).dropna()
        p_less = stats.wilcoxon(dd, alternative="less").pvalue
        ser = TRt.groupby("aligned")[col].mean()
        TRm = TRt[TRt.session.isin(MEAS_NAMES)]
        ser_m = TRm.groupby("aligned")[col].mean()
        temp_m = mc.aligned_series([S for S in MEAS if S["session"] in set(TRm.session)], "temp")[0]["mean"]
        j = ser_m.index.intersection(temp_m.index)
        r, p, n = mc.corr(ser_m.loc[j], temp_m.loc[j])
        print(f"{cls}, {col}: {n_units} units in {n_sess} sessions; mean of trials 1 to 5 {per.early.mean():.3f} against pre {per.pre.mean():.3f} "
              f"(medians {per.early.median():.3f} and {per.pre.median():.3f}; signed-rank P = {p_early:.3g}, one-sided P (early below pre) = {p_less:.3g}, n = {n_}, below in {int((per.early < per.pre).sum())}); "
              f"aligned series against MS temperature (measured sessions, {TRm.groupby(['session', 'u']).ngroups} units): R = {r:.3f}, P = {p:.3g}, n = {n}")
print("expected units: 319 interneurons and 1104 pyramidal cells across 19 sessions (tolerance 32 and 110)")
print("hippocampal temperature: not in the dandiset; the comparison with it is not made")

**Result: reproduced.** Under the raw-rate reading, interneurons (307 units, 23 sessions) and pyramidal cells (1295 units) fire less on aligned trials 1 to 5 than before onset (signed-rank P = 1.9e-6 and 6.3e-16), and the aligned rate series correlate positively with MS temperature (R = 0.806, P = 1.1e-28; R = 0.532, P = 4.0e-10). Normalised rates do not give both parts: their medians on trials 1 to 5 are below pre (0.887 and 0.785 against 1.000; one-sided signed-rank P = 1.4e-4 and 2.4e-18), but the means are above (1.08 and 1.28, pulled up by units with low pre rates), and the normalised series do not correlate with temperature (R = 0.113, P = 0.22; R = 0.147, P = 0.11). The interneuron count is within 287 to 351; the pyramidal count, 1295 over 23 sessions, is above 994 to 1214.

## suppl2d-interneuron-oscillation-slows

> The theta oscillation frequency of putative fast spiking interneurons decreased with MS cooling, as did that of pyramidal cells, by about 14% on average, and pyramidal cells oscillated faster than interneurons.

Spikes during running (above 10 cm/s, Figure S2D "during running"): within place fields for pyramidal cells (fields of the Figure 3 and 4 set), across the whole maze (all trial spikes) for interneurons.

In [ ]:
rows = []
for S in SESS:
    if not S["has_units"]:
        continue
    sp = S["sp"]
    st = S["trials"].st_lab.values[sp.trial.values]
    run = sp.speed.values > mc.RUN_SPEED
    for k, U in enumerate(S["units"]):
        if U["cell_type"] in ("Narrow Interneuron", "Wide Interneuron"):
            r = dict(session=S["session"], rat=S["rat"], cls=U["cell_type"], u=k)
            for s in ("Pre", "Cooling"):
                m = (sp.u.values == k) & (st == s) & run
                g = sp[m]
                r[s] = mc.acg_frequency([gg.t.values * 1000 for _, gg in g.groupby("trial")])[0] if len(g) > 10 else np.nan
            rows.append(r)
IF = pd.DataFrame(rows)
# pyramidal: fp of the place fields from running in-field spikes
rows = []
for S in USESS:
    sp = S["sp"]
    st = S["trials"].st_lab.values[sp.trial.values]
    for _, f in F5[F5.session == S["session"]].iterrows():
        r = dict(session=S["session"], rat=S["rat"], cls="Pyramidal Cell", u=f.u)
        for s in ("Pre", "Cooling"):
            m = (sp.u.values == f.u) & (sp.arm.values == f.arm) & (sp.lin.values >= f.lo) & (sp.lin.values < f.hi) & (st == s) & (sp.speed.values > mc.RUN_SPEED)
            g = sp[m]
            r[s] = mc.acg_frequency([gg.t.values * 1000 for _, gg in g.groupby("trial")])[0] if len(g) > 1 else np.nan
        rows.append(r)
PF = pd.DataFrame(rows)
ALLF = pd.concat([IF, PF], ignore_index=True)
rows = []
for lab_, sel in [("a narrow interneurons", ALLF.cls == "Narrow Interneuron"),
                  ("b narrow and wide interneurons", ALLF.cls.isin(["Narrow Interneuron", "Wide Interneuron"])),
                  ("pyramidal (fields)", ALLF.cls == "Pyramidal Cell"),
                  ("pooled: narrow and wide interneurons and pyramidal fields", ALLF.cls.notna())]:
    g = ALLF[sel].dropna(subset=["Pre", "Cooling"])
    dec = 100 * (1 - g.Cooling / g.Pre)
    rows.append(dict(units=lab_, n=len(g), pre_median=g.Pre.median(), cool_median=g.Cooling.median(),
                     a_median_dec=dec.median(), b_dec_of_mean=100 * (1 - g.Cooling.mean() / g.Pre.mean()), c_mean_dec=dec.mean(),
                     P=mc.signrank(g.Cooling, g.Pre)[0]))
D2 = pd.DataFrame(rows)
print(D2.to_string(index=False))
print(f"interneurons dropped for no autocorrelogram peak in a state: {IF[['Pre', 'Cooling']].isna().any(axis=1).sum()} of {len(IF)}; "
      f"pyramidal fields dropped: {PF[['Pre', 'Cooling']].isna().any(axis=1).sum()} of {len(PF)}")

**Result: partially reproduced.** The oscillation frequency of narrow waveform interneurons falls during cooling (median 8.93 to 8.40 Hz, signed-rank P = 6.4e-13, 145 units), and pyramidal cells oscillate faster than interneurons before cooling (median 10.42 against 8.93 Hz), but the decrease is 6.4 to 8.7% for each class and 7.2 to 8.6% pooled, not 12.6 to 15.4%.

## fig3hi-population-vector-unchanged

> The spatial scale of the population vector correlation of pyramidal cells was not affected by MS cooling.

Per session, arm and state: population vectors of the pyramidal cells' smoothed rate maps (5 cm bins) along the linearized maze; the decorrelation curve is the mean Pearson correlation of vectors d bins apart; the scale is the distance (interpolated) where it first drops to 0.5. The session's scale is the mean of its two arms.

In [ ]:
rows = []
for S in USESS:
    edges = np.arange(0, 355, 5.0)
    pyr = [k for k, U in enumerate(S["units"]) if U["cell_type"] == "Pyramidal Cell"]
    sp = S["sp"]
    st_tr = S["trials"].st_lab.values
    st = st_tr[sp.trial.values]
    r = dict(session=S["session"], rat=S["rat"], n_pyr=len(pyr))
    for s in mc.STATES:
        sc = []
        for arm in ("Left", "Right"):
            occ = mc.occupancy(S, arm, edges, st_tr == s)
            M = []
            for k in pyr:
                cnt, _ = np.histogram(sp.lin[(sp.u.values == k) & (sp.arm.values == arm) & (st == s)], bins=edges)
                raw = np.where(occ > 0, cnt / np.where(occ > 0, occ, 1), np.nan)
                M.append(mc.smooth_map(raw, occ))
            M = np.array(M)
            good = np.isfinite(M).all(0)
            M = M[:, good]
            nb = M.shape[1]
            curve = []
            for dd in range(0, min(nb, 40)):
                cs = [np.corrcoef(M[:, i], M[:, i + dd])[0, 1] for i in range(nb - dd)]
                curve.append(np.nanmean(cs))
            curve = np.array(curve)
            below = np.where(curve < 0.5)[0]
            if len(below) and below[0] > 0:
                j = below[0]
                x = (j - 1) + (curve[j - 1] - 0.5) / (curve[j - 1] - curve[j])
                sc.append(5.0 * x)
            else:
                sc.append(np.nan)
        r[s] = np.nanmean(sc)
    rows.append(r)
PV = pd.DataFrame(rows)
print(PV.round(2).to_string(index=False))
p, n = mc.signrank(PV.Cooling, PV.Pre)
print(f"scale (cm) median pre {PV.Pre.median():.2f}, cooling {PV.Cooling.median():.2f}, post {PV.Post.median():.2f}; "
      f"median change cooling - pre {np.median(PV.Cooling - PV.Pre):.2f} cm; signed-rank P = {p:.3g} (n = {n} sessions)")

**Result: reproduced.** The distance at which the population vector correlation falls to 0.5 does not change with cooling (median 24.0 cm pre, 25.9 cm cooling, median change -0.05 cm, signed-rank P = 0.73 over 23 sessions).

## Place cell pairs (Figure 5)

Fields by the Figure 5 rule (5 cm bins, `lab` states). Pairs are overlapping fields of different cells on the same arm (Methods, "individual overlapping fields"). Spikes as the animal passes through the fields: reading (a) each cell's spikes in its own field, reading (b) both cells' spikes in the union of the two fields. Per pair and state, the cross-correlogram (1 ms bins, lags of plus and minus 400 ms) is convolved with a 60-bin Gaussian window, and the phase cross-correlogram (unwrapped theta phase, 0.01 pi bins over plus and minus 6 pi, for spike pairs within 400 ms) with a 60-bin window (Methods). A pair is used in a state when its cross-correlogram holds at least 20 spike pairs within 200 ms (standard choice; the paper gives no minimum). Summed cross-correlograms sum the pairs' cross-correlograms, each scaled to unit sum (standard choice). The period of a summed cross-correlogram, the spacing of its peaks, is measured as the period of its largest Fourier component (50 to 200 ms; pi to 4 pi on the phase axis) after the slow trend is removed (standard choice; a peak-picking rule gave unstable spacings on the noisier per-session sums). For the pooled sums the spacing of the peaks themselves is printed too (`peak_spacing`).

In [ ]:
T0 = time.time()
LAGS = np.arange(-400, 401)
PH_BIN = 0.01 * np.pi
PH_MAX = 6 * np.pi
PH_LAGS = np.arange(-int(round(PH_MAX / PH_BIN)), int(round(PH_MAX / PH_BIN)) + 1) * PH_BIN


def pair_ccgs(S, FF, reading):
    sp = S["sp"]
    st = S["trials"].st_lab.values[sp.trial.values]
    has_ph = S["has_lfp"]
    # spikes of each cell on each arm, held once
    by = {}
    for (u, arm), g in sp.assign(state=st).groupby(["u", "arm"]):
        by[(u, arm)] = (g.lin.values, g.t.values * 1000, g.phase_u.values if has_ph else None, g.state.values)
    rows = []
    for arm in ("Left", "Right"):
        Fa = FF[(FF.arm == arm)].reset_index(drop=True)
        for i in range(len(Fa)):
            for j in range(i + 1, len(Fa)):
                f1, f2 = Fa.iloc[i], Fa.iloc[j]
                if f1.u == f2.u or f1.hi <= f2.lo or f2.hi <= f1.lo:
                    continue
                # order the pair so that Ltr (peak 2 - peak 1) is signed consistently
                if f2.peak_pos < f1.peak_pos:
                    f1, f2 = f2, f1
                if reading == "a":
                    b1, b2 = (f1.lo, f1.hi), (f2.lo, f2.hi)
                else:
                    b1 = b2 = (min(f1.lo, f2.lo), max(f1.hi, f2.hi))
                l1, t1, p1, s1 = by[(f1.u, arm)]
                l2, t2, p2, s2 = by[(f2.u, arm)]
                m1 = (l1 >= b1[0]) & (l1 < b1[1])
                m2 = (l2 >= b2[0]) & (l2 < b2[1])
                r = dict(session=S["session"], rat=S["rat"], arm=arm, u1=f1.u, u2=f2.u, Ltr=f2.peak_pos - f1.peak_pos)
                for s in ("Pre", "Cooling", "Post"):
                    k1, k2 = m1 & (s1 == s), m2 & (s2 == s)
                    if has_ph:
                        d, dph = mc.pair_diffs(t1[k1], t2[k2], 400, p1[k1], p2[k2])
                    else:
                        d = mc.pair_diffs(t1[k1], t2[k2], 400)
                    h = np.bincount(np.clip(np.round(d).astype(int) + 400, 0, 800), minlength=801).astype(float)
                    r[f"n_{s}"] = h[200:601].sum()
                    r[f"ccg_{s}"] = mc.smooth60(h)
                    if has_ph:
                        kk = np.clip(np.round(dph / PH_BIN).astype(int) + (len(PH_LAGS) // 2), 0, len(PH_LAGS) - 1)
                        r[f"pccg_{s}"] = mc.smooth60(np.bincount(kk, minlength=len(PH_LAGS)).astype(float))
                rows.append(r)
    return rows


def ccg_period(y, kind="time"):
    """Period of a summed cross-correlogram (mc.dominant_period): 50 to 200 ms on
    the time axis, pi to 4 pi on the phase axis; the trend is a Gaussian of 50 ms
    (time) or 0.5 pi (phase)."""
    if kind == "time":
        return mc.dominant_period(y, LAGS, 50, 200, 50)
    return mc.dominant_period(y, PH_LAGS, np.pi, 4 * np.pi, 50)


PAIRS = {}
F5all = FIELDS[(5.0, "lab")]
for reading in ("a", "b"):
    rows = []
    for S in USESS:
        FF = F5all[F5all.session == S["session"]]
        rows += pair_ccgs(S, FF, reading)
    PAIRS[reading] = pd.DataFrame(rows)
print(f"pair cross-correlograms in {time.time() - T0:.0f} s")
for k, P in PAIRS.items():
    use = {s: (P[f"n_{s}"] >= 20) for s in mc.STATES}
    print(f"reading {k}: {len(P)} overlapping pairs in {P.session.nunique()} sessions; usable pre {use['Pre'].sum()}, cooling {use['Cooling'].sum()}, "
          f"post {use['Post'].sum()}, all three {(use['Pre'] & use['Cooling'] & use['Post']).sum()}; sessions with usable pairs in every state "
          f"{P[use['Pre'] & use['Cooling'] & use['Post']].session.nunique()}")
    print("   pairs by rat (usable in pre):", P[use["Pre"]].groupby("rat").size().to_dict())


def summed(P, states, kind="ccg"):
    tot = 0
    n = 0
    for s in states:
        ok = P[f"n_{s}"] >= 20
        for v in P.loc[ok, f"{kind}_{s}"]:
            if v.sum() > 0:
                tot = tot + v / v.sum()
                n += 1
    return (tot if n else None), n


def peak_spacing(y, lags=LAGS, lim=300):
    """Spacing of the peaks of a summed cross-correlogram (the criterion's way (i)).
    The slow trend (a Gaussian of 50 ms, as in ccg_period) is removed, and peaks
    are local maxima within +-lim ms with a prominence of at least 10% of the
    detrended range there, so that small shoulders are not counted (standard
    choices). Returns the mean distance between successive peaks, the distance
    from the peak nearest zero to its neighbours, and the peak lags."""
    from scipy.signal import find_peaks
    from scipy.ndimage import gaussian_filter1d
    if y is None:
        return np.nan, np.nan, []
    z = y - gaussian_filter1d(y, 50)
    w = np.abs(lags) <= lim
    pk = find_peaks(z, prominence=0.1 * (z[w].max() - z[w].min()))[0]
    pk = pk[np.abs(lags[pk]) <= lim]
    if len(pk) < 2:
        return np.nan, np.nan, list(lags[pk])
    j = int(np.argmin(np.abs(lags[pk])))
    nbr = [lags[pk[j]] - lags[pk[j - 1]]] if j > 0 else []
    nbr += [lags[pk[j + 1]] - lags[pk[j]]] if j + 1 < len(pk) else []
    return float(np.mean(np.diff(lags[pk]))), float(np.mean(nbr)), [int(v) for v in lags[pk]]


def lfp_period_ms(S, states):
    tr = S["trials"]
    return 1000 * np.mean(1 / tr.f_hil[tr.st_lab.isin(states)]) if S["has_lfp"] else np.nan


SBYNAME = {S["session"]: S for S in SESS}

## fig5d-ccg-period

> The theta-repeated cross-correlograms of place cell pairs had a period of about 85 ms before cooling, shorter than the LFP theta period (about 110 ms), and the period lengthened during MS cooling in each session (P = 0.002), while the period of the phase cross-correlograms was unaffected (P = 0.32).

In [ ]:
PER = {}
for k, P in PAIRS.items():
    for pre_states, lab_ in [(["Pre"], "pre"), (["Pre", "Post"], "pre and post")]:
        y, n = summed(P, pre_states)
        per = ccg_period(y)
        lfp = np.nanmean([lfp_period_ms(SBYNAME[s_], pre_states) for s_ in P.session.unique()])
        sp_all, sp_c, pk_l = peak_spacing(y)
        print(f"reading {k}, all pairs pooled, {lab_}: {n} pair cross-correlograms, period {per:.1f} ms (Fourier); peak spacing {sp_all:.1f} ms "
              f"(mean over peaks within 300 ms), {sp_c:.1f} ms (central peak to its neighbours), peaks at {pk_l} ms; mean LFP theta period {lfp:.1f} ms")
    rows = []
    for sess, g in P.groupby("session"):
        r = dict(session=sess, rat=g.rat.iloc[0])
        for cond, states in [("control", ["Pre", "Post"]), ("cooling", ["Cooling"]), ("pre", ["Pre"])]:
            y, n = summed(g, states)
            r[f"n_{cond}"] = n
            r[f"T_{cond}"] = ccg_period(y)
            yp, _ = summed(g, states, "pccg") if SBYNAME[sess]["has_lfp"] else (None, 0)
            r[f"Tph_{cond}"] = ccg_period(yp, "phase")
        rows.append(r)
    PS = pd.DataFrame(rows)
    PER[k] = PS
    every = set(P[(P.n_Pre >= 20) & (P.n_Cooling >= 20) & (P.n_Post >= 20)].session)
    for set_name, sel in [("a all sessions with pairs", PS.n_cooling > 0), ("b sessions with pairs in every state", PS.session.isin(every))]:
        d = PS[sel].dropna(subset=["T_control", "T_cooling"])
        p, n = mc.signrank(d.T_cooling, d.T_control)
        dp = PS[sel].dropna(subset=["Tph_control", "Tph_cooling"])
        pp, npp = mc.signrank(dp.Tph_cooling, dp.Tph_control)
        print(f"  reading {k}, sessions {set_name}: n = {n}; period control median {d.T_control.median():.1f} ms, cooling {d.T_cooling.median():.1f} ms; "
              f"longer in {(d.T_cooling > d.T_control).sum()} of {n}; signed-rank P = {p:.3g}; phase period control {dp.Tph_control.median() / np.pi:.3f} pi, "
              f"cooling {dp.Tph_cooling.median() / np.pi:.3f} pi, P = {pp:.3g} (n = {npp})")
    print(PS.round(2).to_string(index=False))

**Result: partially reproduced.** Under pair reading (a), the summed pre cross-correlogram of 9482 pairs has a period of 108.1 ms, shorter than the LFP theta period of 120.9 ms but outside 76.5 to 93.5 ms. Per session the period lengthens during cooling (108.3 to 120.0 ms, longer in 19 of 23 sessions, P = 0.0049), and the phase period does not change (1.86 against 1.84 pi, P = 0.31). Reading (b) gives 104.5 ms and P = 0.042. The spacing of the peaks of the summed pre cross-correlogram, measured directly, is 107.2 ms (mean over 5 peaks within 300 ms; 102.5 ms from the central peak to its neighbours) under pair reading (a) and 103.5 ms under (b), also outside 76.5 to 93.5 ms.

The second way of the criterion, the parameter b of the surface fit, is in fig5cg-compression below.

## fig5e-phase-period-below-2pi

> The period of the summed phase cross-correlograms of place cell pairs was shorter than one theta cycle (2 pi), as place cells oscillate faster than LFP theta.

In [ ]:
for k, P in PAIRS.items():
    for states, lab_ in [(["Pre"], "pre"), (["Cooling"], "cooling")]:
        y, n = summed(P[P.session.map(lambda s_: SBYNAME[s_]["has_lfp"])], states, "pccg")
        per = ccg_period(y, "phase")
        print(f"reading {k}, {lab_}: {n} pairs, phase period {per / np.pi:.3f} pi rad (below 2 pi: {per < 2 * np.pi})")

**Result: reproduced.** The summed phase cross-correlograms have periods of 1.87 pi (pre) and 1.91 pi (cooling) under pair reading (a), and 1.83 and 1.86 pi under reading (b), all below 2 pi.

## fig5h-time-lags-increase, fig5i-phase-lags-unchanged

Pair lag: (a) the largest peak within half the LFP theta period of zero lag in that state (within pi for phase); (b) the peak nearest zero. Absolute values. Signed-rank over pairs usable in both states.

In [ ]:
def pair_lag(y, lags, half):
    pk = mc.peaks(y)
    if len(pk) == 0:
        return np.nan, np.nan
    near = pk[np.abs(lags[pk]) <= half]
    la = lags[near[np.argmax(y[near])]] if len(near) else np.nan
    lb = lags[pk[np.argmin(np.abs(lags[pk]))]]
    return la, lb


LAG = {}
for k, P in PAIRS.items():
    rows = []
    for _, p in P.iterrows():
        S = SBYNAME[p.session]
        r = dict(session=p.session, rat=p.rat, Ltr=p.Ltr)
        for s in mc.STATES:
            ok = p[f"n_{s}"] >= 20
            half = lfp_period_ms(S, [s]) / 2 if S["has_lfp"] else 60.0
            la, lb = pair_lag(p[f"ccg_{s}"], LAGS, half) if ok else (np.nan, np.nan)
            r[f"ta_{s}"], r[f"tb_{s}"] = la, lb
            if S["has_lfp"] and ok:
                pa, pb = pair_lag(p[f"pccg_{s}"], PH_LAGS, np.pi)
            else:
                pa = pb = np.nan
            r[f"pa_{s}"], r[f"pb_{s}"] = pa, pb
        rows.append(r)
    LAG[k] = pd.DataFrame(rows)

In [ ]:
for k, L in LAG.items():
    for rd in ("ta", "tb"):
        out = []
        for a_, b_ in [("Pre", "Cooling"), ("Post", "Cooling"), ("Pre", "Post")]:
            x, y = L[f"{rd}_{a_}"].abs(), L[f"{rd}_{b_}"].abs()
            ok = x.notna() & y.notna()
            p, n = mc.signrank(y[ok], x[ok])
            pr = mc.ranksum(L[f"{rd}_{b_}"].abs(), L[f"{rd}_{a_}"].abs())
            out.append(f"{a_} {x[ok].median():.1f} vs {b_} {y[ok].median():.1f} ms, signed-rank P = {p:.3g} (n = {n}), rank-sum over all pairs P = {pr:.3g}")
        print(f"pairs {k}, lag {rd}: " + "; ".join(out))

**Result: partially reproduced.** Under lag reading (a), pair time lags are longer during cooling than before (27 to 30 ms, signed-rank P = 1.9e-31 over 8506 pairs) and after (P = 9.3e-45), but pre and post also differ (P = 0.034; 0.0082 with pair reading (b)), so no reading gives P >= 0.05 for pre against post. The rank-sum variant over all pairs gives P = 0.16 for pre against post.

In [ ]:
for k, L in list(LAG.items()) + [(k + " without MS13-171130", L[~L.session.isin(BAD_THETA)]) for k, L in LAG.items()]:
    for rd in ("pa", "pb"):
        out = []
        for a_, b_ in [("Pre", "Cooling"), ("Cooling", "Post"), ("Pre", "Post")]:
            x, y = L[f"{rd}_{a_}"].abs(), L[f"{rd}_{b_}"].abs()
            ok = x.notna() & y.notna()
            p, n = mc.signrank(y[ok], x[ok])
            out.append(f"{a_} {x[ok].median():.3f} vs {b_} {y[ok].median():.3f} rad, P = {p:.3g} (n = {n})")
        print(f"pairs {k}, phase lag {rd}: " + "; ".join(out))

**Result: reproduced (assumed).** Under lag reading (a), the largest peak within pi of zero, phase lags do not differ between any two states (P = 0.37, 0.41 and 0.50 over 8271 to 9199 pairs; 0.068 to 0.59 with pair reading (b)). Under reading (b), the peak nearest zero, pre differs from cooling (P = 2.7e-8). The outcome rests on reading (a). Without MS13-171130 (theta channel marked bad), reading (a) gives P = 0.37, 0.62 and 0.71 with pair reading (a), and P = 0.040 for pre against cooling with pair reading (b).

## fig5cg-compression

> Distance-time compression of place cell pairs (0.54 cm/ms before cooling) decreased during MS cooling (P = 0.02), while distance-phase compression was unchanged (P = 0.92).

The density of Figure 5C: the pairs' smoothed cross-correlograms (each scaled to unit sum) averaged in bins of Ltr (5 cm) at each lag (1 ms), over Ltr of plus and minus 100 cm and lags of plus and minus 400 ms. The surface of the Methods is fitted by least squares (scipy curve_fit), x the distance Ltr and y the lag, reading (a) Z = e (cos(2 pi x / (a b) + 2 pi y / b) + 1) exp(-(x + f y)^2 / 2c^2) exp(-(y - g)^2 / 2d^2) and reading (b) with the +1 inside the cosine. Start values: a from the main lag of each Ltr bin of the pooled pre density (within 40 cm), b the period of the summed cross-correlogram, c 60 cm, d 150 ms (standard choices). The fit's R2 against the density is printed; a negative R2 means the surface fits the density worse than its mean. The compression is |a| (cm/ms on the time axis, cm/rad on the phase axis). Per session the same fit on that session's pairs, control (pre and post) and cooling.

In [ ]:
from scipy.optimize import curve_fit


def surf(XY, a, b, c, d, e, f, g, inside=False):
    x, y = XY
    if inside:
        osc = np.cos(2 * np.pi * x / (a * b) + 2 * np.pi * y / b + 1)
    else:
        osc = np.cos(2 * np.pi * x / (a * b) + 2 * np.pi * y / b) + 1
    return e * osc * np.exp(-(x + f * y) ** 2 / (2 * c ** 2)) * np.exp(-(y - g) ** 2 / (2 * d ** 2))


def density(P, states, kind="ccg", lags=LAGS, xbin=5.0, xmax=100.0):
    xb = np.arange(-xmax, xmax + xbin, xbin)
    Z = np.zeros((len(xb) - 1, len(lags)))
    N = np.zeros(len(xb) - 1)
    for s in states:
        ok = (P[f"n_{s}"] >= 20) & P[f"{kind}_{s}"].map(lambda v: isinstance(v, np.ndarray))
        for L, v in zip(P.Ltr[ok], P[f"{kind}_{s}"][ok]):
            i = np.searchsorted(xb, L, side="right") - 1
            if 0 <= i < len(N) and v.sum() > 0:
                Z[i] += v / v.sum()
                N[i] += 1
    with np.errstate(invalid="ignore"):
        Z = Z / N[:, None]
    xc = (xb[:-1] + xb[1:]) / 2
    return xc, Z, N


def a_start(P, states, kind="ccg"):
    """Start value of the compression: from the density, the main lag (largest
    peak within half a theta cycle) of each Ltr bin within 40 cm, regressed on
    Ltr; the ridges of the surface have slope -1/a."""
    lags = LAGS if kind == "ccg" else PH_LAGS
    half = 60.0 if kind == "ccg" else np.pi
    xc, Z, N = density(P, states, kind, lags)
    w = np.abs(lags) <= half
    xs = [xc[i] for i in range(len(xc)) if N[i] >= 5 and abs(xc[i]) <= 40]
    ys = [lags[w][np.argmax(Z[i][w])] for i in range(len(xc)) if N[i] >= 5 and abs(xc[i]) <= 40]
    if len(xs) < 3:
        return np.nan
    return -1 / stats.linregress(xs, ys).slope


def fit_surface(P, states, kind="ccg", inside=False, a0=None, b0=None):
    lags = LAGS if kind == "ccg" else PH_LAGS
    xc, Z, N = density(P, states, kind, lags)
    ok = N > 0
    if ok.sum() < 5:
        return dict(a=np.nan, b=np.nan, ok=False)
    X, Y = np.meshgrid(xc[ok], lags, indexing="ij")
    XY = np.vstack([X.ravel(), Y.ravel()])
    z = Z[ok].ravel()
    z = z - np.nanmin(z)
    d0 = 150.0 if kind == "ccg" else 150.0 * 2 * np.pi / 120.0
    p0 = [a0, b0, 60.0, d0, np.nanmax(z) / 2, 0.0, 0.0]
    sst = np.sum((z - z.mean()) ** 2)
    # the envelope alone (the two Gaussian factors, no oscillation): how much of the density a smooth surface explains
    try:
        pe, _ = curve_fit(lambda XY, c, d, e, f, g: e * np.exp(-(XY[0] + f * XY[1]) ** 2 / (2 * c ** 2)) * np.exp(-(XY[1] - g) ** 2 / (2 * d ** 2)),
                          XY, z, p0=[60.0, d0, np.nanmax(z), 0.0, 0.0], maxfev=3000)
        c, d, e, f, g = pe
        r2_env = 1 - np.sum((z - e * np.exp(-(XY[0] + f * XY[1]) ** 2 / (2 * c ** 2)) * np.exp(-(XY[1] - g) ** 2 / (2 * d ** 2))) ** 2) / sst
    except Exception:
        r2_env = np.nan
    try:
        p, _ = curve_fit(lambda XY, *q: surf(XY, *q, inside=inside), XY, z, p0=p0, maxfev=3000)
        resid = z - surf(XY, *p, inside=inside)
        r2 = 1 - np.sum(resid ** 2) / sst
        return dict(a=abs(p[0]), b=abs(p[1]), r2=r2, r2_env=r2_env, ok=True, p=p, err="")
    except Exception as ex:
        return dict(a=np.nan, b=np.nan, r2=np.nan, r2_env=r2_env, ok=False, err=str(ex))

In [ ]:
T0 = time.time()
CMP = {}
for k, P in PAIRS.items():
    L = LAG[k]
    ok = L.ta_Pre.notna()
    ls = stats.linregress(L.ta_Pre[ok], L.Ltr[ok])
    okp = L.pa_Pre.notna()
    lsp = stats.linregress(L.pa_Pre[okp], L.Ltr[okp])
    yP, _ = summed(P, ["Pre"])
    b0 = ccg_period(yP)
    yPp, _ = summed(P[P.session.map(lambda s_: SBYNAME[s_]["has_lfp"])], ["Pre"], "pccg")
    b0p = ccg_period(yPp, "phase")
    a0t = a_start(P, ["Pre"], "ccg")
    a0p = a_start(P[P.session.map(lambda s_: SBYNAME[s_]["has_lfp"])], ["Pre"], "pccg")
    print(f"pairs {k}: start values a {a0t:.3f} cm/ms and {a0p:.2f} cm/rad; least-squares slope of Ltr on the main lag (pre): {ls.slope:.3f} cm/ms (R = {ls.rvalue:.3f}, n = {ok.sum()}); "
          f"on the phase lag {lsp.slope:.2f} cm/rad; summed period {b0:.1f} ms, {b0p / np.pi:.3f} pi")
    for inside in (False, True):
        fT = fit_surface(P, ["Pre"], "ccg", inside, a0=a0t, b0=b0)
        fP = fit_surface(P[P.session.map(lambda s_: SBYNAME[s_]["has_lfp"])], ["Pre"], "pccg", inside, a0=a0p, b0=b0p)
        print(f"   surface reading {'b (+1 inside)' if inside else 'a'}: all pre pairs pooled, time compression {fT['a']:.3f} cm/ms, period b {fT['b']:.1f} ms "
              f"(R2 {fT['r2']:.3f}; envelope alone R2 {fT['r2_env']:.3f}){' failed: ' + fT['err'] if not fT['ok'] else ''}; "
              f"phase compression {fP['a']:.2f} cm/rad, b {fP['b'] / np.pi:.3f} pi (R2 {fP['r2']:.3f}; envelope alone R2 {fP['r2_env']:.3f}){' failed: ' + fP['err'] if not fP['ok'] else ''}")
        if (k, inside) == ("a", False):
            POOLED_FIT = (P, fT)
        rows = []
        for sess, g in P.groupby("session"):
            has_lfp = SBYNAME[sess]["has_lfp"]
            r = dict(session=sess)
            for cond, states in [("control", ["Pre", "Post"]), ("cooling", ["Cooling"])]:
                f1 = fit_surface(g, states, "ccg", inside, a0=a0t, b0=b0)
                r[f"cT_{cond}"], r[f"bT_{cond}"] = f1["a"], f1["b"]
                r[f"r2T_{cond}"], r[f"r2envT_{cond}"], r[f"errT_{cond}"] = f1.get("r2", np.nan), f1.get("r2_env", np.nan), f1.get("err", "")
                r[f"a0_{cond}"] = a_start(g, states, "ccg")
                if has_lfp:
                    f2 = fit_surface(g, states, "pccg", inside, a0=a0p, b0=b0p)
                    r[f"cP_{cond}"] = f2["a"]
                    r[f"okP_{cond}"] = f2["ok"]
                else:
                    r[f"cP_{cond}"] = np.nan
                Lg = L[L.session == sess]
                cols = ["ta_Pre", "ta_Post"] if cond == "control" else ["ta_Cooling"]
                x = np.concatenate([Lg[c_].values for c_ in cols]); y = np.concatenate([Lg.Ltr.values for _ in cols])
                m = np.isfinite(x)
                r[f"ls_{cond}"] = stats.linregress(x[m], y[m]).slope if m.sum() > 5 else np.nan
            rows.append(r)
        CS = pd.DataFrame(rows)
        CMP[(k, inside)] = CS
        p1, n1 = mc.signrank(CS.cT_cooling, CS.cT_control)
        p2, n2 = mc.signrank(CS.cP_cooling, CS.cP_control)
        p3, n3 = mc.signrank(CS.bT_cooling, CS.bT_control)
        p4, n4 = mc.signrank(CS.ls_cooling, CS.ls_control)
        p5, n5 = mc.signrank(CS.a0_cooling.abs(), CS.a0_control.abs())
        fails = pd.concat([CS.errT_control, CS.errT_cooling])
        fails = fails[fails != ""].str.slice(0, 70).value_counts().to_dict()
        print(f"   per-session time fits converged: control {CS.cT_control.notna().sum()} of {len(CS)}, cooling {CS.cT_cooling.notna().sum()} of {len(CS)}; "
              f"median R2 of converged fits control {CS.r2T_control.median():.3f}, cooling {CS.r2T_cooling.median():.3f} (envelope alone {CS.r2envT_control.median():.3f}, "
              f"{CS.r2envT_cooling.median():.3f}); failures: {fails}; phase fits converged: control {int(CS.get('okP_control', pd.Series(dtype=bool)).fillna(False).sum())}, "
              f"cooling {int(CS.get('okP_cooling', pd.Series(dtype=bool)).fillna(False).sum())}")
        print(f"   density ridge slope (start value of a) per session, a variant: control median {CS.a0_control.abs().median():.3f}, cooling {CS.a0_cooling.abs().median():.3f} cm/ms, "
              f"lower in cooling in {int((CS.a0_cooling.abs() < CS.a0_control.abs()).sum())} of {n5}, signed-rank P = {p5:.3g}")
        d1 = (CS.cT_cooling - CS.cT_control)
        print(f"   per session ({n1}): time compression control median {CS.cT_control.median():.3f}, cooling {CS.cT_cooling.median():.3f} cm/ms, "
              f"lower in {(d1 < 0).sum()}, P = {p1:.3g}; phase compression control {CS.cP_control.median():.2f}, cooling {CS.cP_cooling.median():.2f} cm/rad, "
              f"P = {p2:.3g} (n = {n2}); fitted period b control {CS.bT_control.median():.1f}, cooling {CS.bT_cooling.median():.1f} ms, P = {p3:.3g}; "
              f"least-squares slope control {CS.ls_control.median():.3f}, cooling {CS.ls_cooling.median():.3f}, P = {p4:.3g}")
print(f"({time.time() - T0:.0f} s)")
print(CMP[("a", False)].drop(columns=[c for c in CMP[("a", False)] if c.startswith("err")]).round(3).to_string(index=False))
# pooled pre density (pair reading a) and the fitted surface of reading (a)
Pp, fT = POOLED_FIT
xc, Z, N = density(Pp, ["Pre"])
fig, ax = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
ax[0].imshow(Z, aspect="auto", origin="lower", extent=[LAGS[0], LAGS[-1], xc[0], xc[-1]])
ax[0].set_title("pooled pre density"); ax[0].set_xlabel("lag (ms)"); ax[0].set_ylabel("Ltr (cm)")
if fT["ok"]:
    X, Y = np.meshgrid(xc, LAGS, indexing="ij")
    ax[1].imshow(surf(np.vstack([X.ravel(), Y.ravel()]), *fT["p"]).reshape(X.shape), aspect="auto", origin="lower", extent=[LAGS[0], LAGS[-1], xc[0], xc[-1]])
ax[1].set_title(f"fitted surface, reading (a), R2 {fT['r2']:.2f}"); ax[1].set_xlabel("lag (ms)")
plt.tight_layout(); plt.show()
print(f"density: {int((N > 0).sum())} of {len(N)} Ltr bins hold pairs; values {np.nanmin(Z):.2e} to {np.nanmax(Z):.2e}; the full grid is shown")

**Result: blocked (methods_underspecified).** The surface of the Methods, under both readings of its equation, does not describe the density: the converged fits explain less than the density's mean (pooled pre R2 -0.135 under reading (a); per session median R2 -0.10 to -5.1), while its envelope alone (the two Gaussian factors without the oscillation) has a median R2 of 0.41 to 0.68 per session. The fits fail to converge in most cases (reading (a), pair reading (a): 9 of 23 sessions in each condition, 5 in both; reading (b): 2 and 6), always for reaching the evaluation limit. The fitted a therefore does not measure the slope of the density's ridges, and the paper gives no grid, start values or fitting procedure from which a working fit could be taken. As variants, none of which is the criterion's measure: the density's ridge slope is 0.556 cm/ms pooled (0.561 with pair reading b) and does not fall with cooling per session (0.566 against 0.628 cm/ms, P = 0.62; 0.623 against 0.616, P = 0.71); where the time fits converge in both conditions, control and cooling do not differ (P = 0.81, 5 sessions; P = 1.0, 10 sessions); the least-squares slope of Ltr on the main lag does not differ (P = 0.67 and 0.36).

## fig5-speed-control-cooling

> In the sessions of Figure 5, mean running speed was 87.3 +- 8.0 in control trials (pre and post combined) and 86.2 +- 9.9 during MS cooling.

In [ ]:
fig5_sessions = sorted(set(PAIRS["a"].session[PAIRS["a"].n_Pre >= 20]))
every = sorted(set(PAIRS["a"].session[(PAIRS["a"].n_Pre >= 20) & (PAIRS["a"].n_Cooling >= 20) & (PAIRS["a"].n_Post >= 20)]))
for set_name, names in [("sessions with pairs", fig5_sessions), ("sessions with pairs in every state", every)]:
    for c in ["speed_all", "speed_run"]:
        ctrl, cool = [], []
        for nm in names:
            tr = SBYNAME[nm]["trials"]
            ctrl.append(tr[c][tr.st_lab.isin(["Pre", "Post"])].mean())
            cool.append(tr[c][tr.st_lab == "Cooling"].mean())
        ctrl, cool = np.array(ctrl), np.array(cool)
        print(f"{set_name} (n = {len(names)}), {c}: control {ctrl.mean():.1f} +- {ctrl.std(ddof=1):.1f}, cooling {cool.mean():.1f} +- {cool.std(ddof=1):.1f} cm/s; "
              f"within tolerance: control {abs(ctrl.mean() - 87.3) <= 8.0}, cooling {abs(cool.mean() - 86.2) <= 9.9}")

**Result: reproduced.** In the 23 sessions with place cell pairs, mean speed is 80.3 +- 9.2 cm/s in control trials and 77.1 +- 12.5 cm/s during cooling (81.0 and 78.1 above 10 cm/s), within 79.3 to 95.3 and 76.3 to 96.1.

## fig5j-fraction-active-unchanged, fig5k-spikes-per-cycle-unchanged

Theta cycles during the trials of the sessions with place cell pairs (as suppl3-theta-cycle-duration). Per cycle, the fraction of the session's place cells (cells with a Figure 5 field) with at least one spike, the same fraction of all curated units (the legend's variant), and the spikes of all curated units.

In [ ]:
rows = []
for nm in fig5_sessions:
    S = SBYNAME[nm]
    if not S["has_lfp"]:
        continue
    cyc = S["cycles"]
    sp = S["sp"]
    pcs = set(F5all[F5all.session == nm].u)
    t = sp.t.values
    ci = np.searchsorted(cyc.t0.values, t, side="right") - 1
    ok = (ci >= 0) & (t < cyc.t1.values[np.clip(ci, 0, None)])
    ci, uu = ci[ok], sp.u.values[ok]
    nsp = np.bincount(ci, minlength=len(cyc))
    pair = pd.DataFrame(dict(c=ci, u=uu)).drop_duplicates()
    act_all = np.bincount(pair.c, minlength=len(cyc))
    pp = pair[pair.u.isin(pcs)]
    act_pc = np.bincount(pp.c, minlength=len(cyc))
    st = S["trials"].st_lab.values[cyc.trial.values]
    rows.append(pd.DataFrame(dict(session=nm, state=st, frac_pc=act_pc / max(len(pcs), 1),
                                  frac_all=act_all / len(S["units"]), spikes=nsp)))
CY = pd.concat(rows)
print(f"{len(CY)} cycles in {CY.session.nunique()} sessions; per state {CY.state.value_counts().to_dict()}")

In [ ]:
for c in ["frac_pc", "frac_all"]:
    out = []
    for a_, b_ in [("Pre", "Cooling"), ("Post", "Cooling"), ("Pre", "Post")]:
        x, y = CY[c][CY.state == a_], CY[c][CY.state == b_]
        out.append(f"{a_} {x.mean():.4f} vs {b_} {y.mean():.4f} (medians {x.median():.4f}, {y.median():.4f}), rank-sum P = {mc.ranksum(x, y):.3g}")
    print(f"{c}: " + "; ".join(out))
per = CY.groupby(["session", "state"]).frac_pc.mean().unstack()
print(f"per session mean fraction, cooling - pre: median {np.median(per.Cooling - per.Pre):.4f}, signed-rank P = {mc.signrank(per.Cooling, per.Pre)[0]:.3g} (n = {len(per)})")

**Result: not reproduced.** Over 82574 theta cycles in 22 sessions, the fraction of place cells active per cycle is higher during cooling than before (0.200 against 0.189, rank-sum P = 1.6e-25). Averaged per session first, it does not differ (median change -0.0007, signed-rank P = 1.0, a variant).

In [ ]:
out = []
for a_, b_ in [("Pre", "Cooling"), ("Cooling", "Post"), ("Pre", "Post")]:
    x, y = CY.spikes[CY.state == a_], CY.spikes[CY.state == b_]
    out.append(f"{a_} {x.mean():.2f} vs {b_} {y.mean():.2f} spikes per cycle (medians {x.median():.0f}, {y.median():.0f}), rank-sum P = {mc.ranksum(x, y):.3g}")
print("; ".join(out))
per = CY.groupby(["session", "state"]).spikes.mean().unstack()
print(f"per session mean, cooling - pre: median {np.median(per.Cooling - per.Pre):.3f}, signed-rank P = {mc.signrank(per.Cooling, per.Pre)[0]:.3g} (n = {len(per)})")

**Result: not reproduced.** Spikes per theta cycle differ between all pairs of states by rank-sum over cycles (pre 30.8, cooling 31.1, post 35.2; P = 4.9e-7, 1.8e-39 and 5.8e-65). Per session means do not differ from pre to cooling (P = 0.80, a variant).

## suppl1g-waveforms-unchanged

> The waveforms of hippocampal action potentials were not affected by MS cooling.

The dandiset holds no unit waveforms, only each unit's largest channel. Mean waveforms are taken here from the 20 kHz wide-band signal, high-pass filtered at 300 Hz, on a subset: one session per rat with typed units and the most units (MS10 sessions have few units and are left out), and in each session two 52 s blocks in pre, three in cooling and two in post (`mc.unit_waveforms`). Trough-to-peak latency is measured on the 20 kHz samples and, as a variant, on the mean waveform interpolated by a cubic spline to 200 kHz. Reading the whole wide-band signal (3 TB) does not fit the run; one block of 16 channels streams in about 1.2 s.

In [ ]:
T0 = time.time()
WF_SESS = []
for rat in ["MS12", "MS13", "MS21", "MS22"]:
    cand = [S for S in SESS if S["rat"] == rat and S["has_units"]]
    WF_SESS.append(max(cand, key=lambda S: sum(u["maxch"] >= 0 for u in S["units"])))
WF = pd.concat([mc.unit_waveforms(S) for S in WF_SESS], ignore_index=True)
print(f"sessions {[S['session'] for S in WF_SESS]} in {time.time() - T0:.0f} s")
Ww = WF.pivot_table(index=["session", "u", "cell_type"], columns="cond", values=["trough_to_peak_ms", "trough_to_peak_interp_ms", "p2p_uV", "n"])
Ww = Ww[(Ww[("n", "control")] >= 20) & (Ww[("n", "cooling")] >= 20)]
print(f"units with at least 20 spikes in both conditions: {len(Ww)} of {WF[['session', 'u']].drop_duplicates().shape[0]}")
print("spikes per unit in the waveform blocks, median (range): control", Ww[("n", "control")].median(), f"({Ww[('n', 'control')].min():.0f} to {Ww[('n', 'control')].max():.0f}), cooling",
      Ww[("n", "cooling")].median(), f"({Ww[('n', 'cooling')].min():.0f} to {Ww[('n', 'cooling')].max():.0f})")
for m in ["trough_to_peak_ms", "trough_to_peak_interp_ms", "p2p_uV"]:
    c, k = Ww[(m, "control")], Ww[(m, "cooling")]
    p, n = mc.signrank(k, c)
    print(f"{m}: median control {c.median():.3f}, cooling {k.median():.3f}; median change {np.median(k - c):.3f} "
          f"({100 * np.median((k - c) / c):.2f}%); signed-rank P = {p:.3g} (n = {n})")
    for s_, g in Ww.groupby(level="session"):
        print(f"   {s_}: n = {len(g)}, median change {np.median(g[(m, 'cooling')] - g[(m, 'control')]):.3f}, P = {mc.signrank(g[(m, 'cooling')], g[(m, 'control')])[0]:.3g}")

**Result: not reproduced.** On the subset of 4 sessions (263 units), the trough-to-peak latency of the mean waveform is longer during cooling (median change +0.05 ms, signed-rank P = 1.1e-9) and the peak-to-peak amplitude differs slightly (+0.65 uV, +0.7%, P = 0.026). The latency change is significant in 2 of the 4 sessions on its own. Measured on the mean waveforms interpolated to 200 kHz, the latency is still longer during cooling (0.79 against 0.82 ms, median change +0.035 ms, P = 3.1e-11; significant in each of the 4 sessions). The units have a median of 158 spikes in the control blocks and 115 in the cooling blocks (at least 20 each).

## suppl1-hippocampal-temperature

Blocked (`data_missing`): the control experiment with simultaneous MS and hippocampal temperature (Figure S1H to K) is not in the dandiset. No file holds a hippocampal temperature (the behaviour modules list only `MeasuredTemperature` or `ModelledTemperature` of the septal probe; printed below).

In [ ]:
print(sorted(set(k for S in SESS for k in ["MeasuredTemperature" if S["temp_kind"] == "measured" else "ModelledTemperature"])))
print("epoch paradigms in the files:", sorted(set(p for S in SESS if "epochs" in S for p in S["epochs"].behavioral_paradigm)))
print(f"notebook run time {(time.time() - T_START) / 60:.1f} min; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

**Result: blocked (data_missing).** The dandiset holds no hippocampal temperature.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

| claim | outcome |
|---|---|
| session-structure | partially reproduced |
| cooling-time-course | partially reproduced |
| fig2g-theta-freq-temperature | partially reproduced |
| fig2h-theta-power-temperature | reproduced |
| fig2j-frequency-consistent-across-animals | reproduced |
| fig2i-theta-pre-cooling-post | reproduced |
| disc-theta-frequency-12-percent | reproduced |
| fig6d-errors-increase | partially reproduced |
| fig6e-errors-recover | reproduced |
| fig6c-worst-trials-4-to-8 | reproduced |
| fig6f-errors-temperature | partially reproduced |
| fig6g-errors-theta-frequency | partially reproduced |
| fig6h-errors-theta-power | reproduced |
| fig6i-errors-not-speed | not reproduced |
| suppl5j-errors-best-theta-frequency | partially reproduced |
| fig4e-speed-decreases-within-session | reproduced |
| fig5-speed-control-cooling | reproduced |
| suppl2-place-cell-rates-decrease | partially reproduced |
| suppl2ab-rates-correlated | reproduced |
| disc-interneuron-rates-decrease | partially reproduced |
| suppl5gh-rates-follow-temperature | reproduced |
| fig4b-fp-flfp-decrease | reproduced |
| fig4c-fp-flfp-correlation | partially reproduced |
| fig4ac-fp-flfp-difference-preserved | not reproduced |
| fig4a-frequency-temperature-slopes | partially reproduced |
| fig3c-field-size-correlated | not reproduced |
| fig3d-size-fp-negative | partially reproduced |
| fig3e-model-field-size | not reproduced |
| fig3f-information-unchanged | not reproduced |
| fig3hi-population-vector-unchanged | reproduced |
| fig4d-precession-slope-decrease | partially reproduced (assumed) |
| suppl3e-precession-temperature | partially reproduced |
| fig4f-speed-theta-power | reproduced |
| fig4gh-fp-speed | partially reproduced |
| suppl2e-trial-speed-theta-frequency | reproduced |
| suppl3-theta-cycle-duration | partially reproduced |
| fig5d-ccg-period | partially reproduced |
| fig5e-phase-period-below-2pi | reproduced |
| fig5cg-compression | blocked (methods_underspecified) |
| fig5h-time-lags-increase | partially reproduced |
| fig5i-phase-lags-unchanged | reproduced (assumed) |
| fig5j-fraction-active-unchanged | not reproduced |
| fig5k-spikes-per-cycle-unchanged | not reproduced |
| suppl1g-waveforms-unchanged | not reproduced |
| suppl2d-interneuron-oscillation-slows | partially reproduced |
| suppl1-hippocampal-temperature | blocked (data_missing) |

The outcomes of the 20 tested claims on place fields and pairs rest on an assumption: fields are found on the map over all trials of an arm, states pooled (the attempt record marks them `assumed`). The two precession claims also rest on the range of the slope search.

How the analysis differs from the paper's:

- Sessions. The dandiset holds 51 sessions of 5 rats; the paper reports 53. Temperature analyses use the 31 sessions with a measured MS temperature (MS10 has none, MS12 two); the others hold a modelled curve. Place cell analyses use the 23 sessions with typed pyramidal cells; the paper's subsets (21, 19, 18 and 10 sessions) are not named and cannot be matched.
- Place fields of Figures 3 and 4 were drawn by hand in the paper; here they follow the Figure 5 rule with a peak above 10 Hz, on maps over all trials of an arm. This gives 1223 fields and 878 with an fp in every state, against the paper's 365, and 12834 overlapping pairs against 2948.
- Cooling onset is the dataset's time stamp; states are the dataset's labels, with the 10 and 12 min readings as variants. Thirteen sessions have a second cooling epoch; aligned series stop at it.
- Theta is measured on the lab's theta channel; for MS13-171130 that channel is marked bad, and the claims that use theta are also shown without that session. The surface fit of Figure 5 does not describe the density under either reading of its equation, so fig5cg is blocked.
- The phase precession slope is searched over plus and minus 2 cycles per field, an assumption; the paper gives no range, and a range of 1 cycle pins about 30% of the slopes at its bound.
- Spatial information is also shown with equal trial numbers per state.
- Waveforms were measured on 4 sessions from short blocks of the wide-band signal, as the dandiset holds none.

What was found about the dataset:

- The behaviour descriptions say the tracking is at 40 Hz in every session; the timestamps give about 120 Hz in 44 sessions, 40 Hz in 6 and 30 Hz in 1 (the paper says 120 Hz).
- Measured MS temperature traces hold artefacts, values from -1751 C to 1354 C and runs at the 0 and 50 C rails, in at least six sessions.
- The 2018-07-19 sessions (MS21, MS22) are labelled as cooled, but their measured temperature does not fall (minimum 37.4 and 35.2 C).
- No hippocampal temperature, unit waveforms or place fields are in the dandiset, and the lab's theta channel of MS13-171130 is marked bad. MS12-170715 has no LFP. Epoch environments name a linear track for some theta maze epochs.
- 28 sessions hold curated units, 23 of them typed; 23 sessions have none.